In [113]:
#importing all library

import pandas as pd
import numpy as np
import json
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [114]:
# File paths
ARRIVALS = "/content/track3_mandi_arrivals.csv"
MASTER = "/content/track3_mandi_master.csv"
PRICE = "/content/track3_price_and_msp.json"
TRANSPORT = "/content/track3_transport_logistics.csv"
WEATHER = "/content/track3_weather_sensors.xlsx"

# Load datasets
arrivals = pd.read_csv(ARRIVALS)
master = pd.read_csv(MASTER)
transport = pd.read_csv(TRANSPORT)
weather = pd.read_excel(WEATHER)

with open(PRICE, "r", encoding="utf-8") as f:
    price = pd.DataFrame(json.load(f))

print("Arrivals:", arrivals.shape)
print("Master:", master.shape)
print("Price:", price.shape)
print("Transport:", transport.shape)
print("Weather:", weather.shape)

Arrivals: (25750, 8)
Master: (60, 6)
Price: (12000, 9)
Transport: (10400, 10)
Weather: (15000, 7)


In [115]:
datasets = {
    "Arrivals": arrivals,
    "Master": master,
    "Price": price,
    "Transport": transport,
    "Weather": weather
}

for name, df in datasets.items():
    print("\n" + "="*60)
    print(name)
    print("="*60)

    print("Shape:", df.shape)
    print("\nData types:")
    print(df.dtypes)


Arrivals
Shape: (25750, 8)

Data types:
arrival_id           object
date                 object
mandi_id             object
crop_name            object
variety              object
arrival_quantity     object
unit                 object
farmer_count        float64
dtype: object

Master
Shape: (60, 6)

Data types:
mandi_id             object
mandi_name           object
district             object
state                object
mandi_type           object
total_area_acres    float64
dtype: object

Price
Shape: (12000, 9)

Data types:
record_id      object
date           object
mandi_id       object
district       object
crop_name      object
min_price      object
max_price      object
modal_price    object
msp            object
dtype: object

Transport
Shape: (10400, 10)

Data types:
trip_id                  object
mandi_id                 object
destination_warehouse    object
departure_time           object
arrival_time             object
transit_hours            object
distance          

In [116]:
for name, df in datasets.items():
    print("\n" + "="*60)
    print(name)
    print("="*60)

    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)

    audit = pd.DataFrame({
        "missing_count": missing,
        "missing_percent": missing_pct
    })

    print(audit[audit["missing_count"] > 0])


Arrivals
              missing_count  missing_percent
arrival_id              484             1.88
variety                3735            14.50
unit                   5143            19.97
farmer_count           3899            15.14

Master
                  missing_count  missing_percent
district                      4             6.67
state                         4             6.67
mandi_type                   11            18.33
total_area_acres              6            10.00

Price
          missing_count  missing_percent
mandi_id           1235            10.29
district            773             6.44

Transport
               missing_count  missing_percent
arrival_time            1053            10.12
transit_hours            518             4.98
distance_unit           1032             9.92
vehicle_no              1622            15.60
driver_id               1551            14.91

Weather
                  missing_count  missing_percent
timestamp                  1555      

In [117]:
for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

Arrivals: 750 duplicate rows
Master: 3 duplicate rows
Price: 0 duplicate rows
Transport: 400 duplicate rows
Weather: 0 duplicate rows


In [118]:
# Inspect Mandi Master

print(master.head(20))

print("\nUnique mandi IDs:")
print(master["mandi_id"].unique())

print("\nUnique mandi types:")
print(master["mandi_type"].unique())

print("\nUnique districts:")
print(master["district"].unique())

print("\nDuplicate rows:")
print(master[master.duplicated(keep=False)].sort_values("mandi_id"))

    mandi_id             mandi_name       district          state mandi_type  \
0   MANDI001        Hyderabad Mandi       ludhiana         Punjab    Private   
1   MANDI006            Kochi Mandi       Amritsar         Punjab     Direct   
2   MANDI037    Jorhat Grain Market    Kurukshetra        Haryana       APMC   
3   MANDI046       Baranagar Market            NaN  Uttar Pradesh    Private   
4   MANDI014  Bathinda Grain Market        Patiala         Punjab    PRIVATE   
5   MANDI055        Chandigarh APMC       Bareilly  Uttar Pradesh       apmc   
6   MANDI034  Ludhiana Grain Market         Ambala        Haryana        NaN   
7   MANDI049    Machilipatnam Mandi  Muzaffarnagar  Uttar Pradesh    Private   
8   MANDI013      Aurangabad Market        Patiala         Punjab    Private   
9   MANDI001        Hyderabad Mandi       ludhiana         Punjab    Private   
10  MANDI047            Orai Market  Muzaffarnagar  Uttar Pradesh       APMC   
11  MANDI051        Gulbarga Market     

In [119]:
import pandas as pd
import numpy as np

# Keep original untouched
master_clean = master.copy()

# --------------------------------------------------
# 1. Remove exact duplicate rows
# --------------------------------------------------

before = len(master_clean)

master_clean = master_clean.drop_duplicates()

after = len(master_clean)

print(f"Duplicate rows removed: {before - after}")
print(f"Rows before cleaning: {before}")
print(f"Rows after cleaning: {after}")


# --------------------------------------------------
# 2. Standardize Mandi ID
# --------------------------------------------------

def clean_mandi_id(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().upper()

    # Remove separators
    x = x.replace("-", "").replace("_", "").replace(" ", "")

    # Convert M012 -> MANDI012
    if x.startswith("M") and not x.startswith("MANDI"):
        number = x[1:]
        if number.isdigit():
            return f"MANDI{int(number):03d}"

    # Convert MANDI12 -> MANDI012
    if x.startswith("MANDI"):
        number = x[5:]
        if number.isdigit():
            return f"MANDI{int(number):03d}"

    return x


master_clean["mandi_id"] = master_clean["mandi_id"].apply(clean_mandi_id)


# --------------------------------------------------
# 3. Standardize text columns
# --------------------------------------------------

master_clean["mandi_name"] = (
    master_clean["mandi_name"]
    .astype("string")
    .str.strip()
)

master_clean["district"] = (
    master_clean["district"]
    .astype("string")
    .str.strip()
    .str.title()
)

master_clean["state"] = (
    master_clean["state"]
    .astype("string")
    .str.strip()
    .str.title()
)


# --------------------------------------------------
# 4. Standardize mandi type
# --------------------------------------------------

master_clean["mandi_type"] = (
    master_clean["mandi_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

master_clean["mandi_type"] = master_clean["mandi_type"].replace({
    "private": "Private",
    "apmc": "APMC",
    "direct": "Direct"
})

master_clean["mandi_type"] = master_clean["mandi_type"].fillna("Unknown")


# --------------------------------------------------
# 5. Clean area
# --------------------------------------------------

master_clean["total_area_acres"] = pd.to_numeric(
    master_clean["total_area_acres"],
    errors="coerce"
)


# --------------------------------------------------
# 6. Check result
# --------------------------------------------------

print("\nCleaned Mandi Master:")
display(master_clean.head(20))

print("\nMandi types:")
print(master_clean["mandi_type"].value_counts(dropna=False))

print("\nDistricts:")
print(sorted(master_clean["district"].dropna().unique()))

print("\nMissing values:")
print(master_clean.isna().sum())

print("\nDuplicate mandi IDs:")
print(master_clean["mandi_id"].duplicated().sum())

Duplicate rows removed: 3
Rows before cleaning: 60
Rows after cleaning: 57

Cleaned Mandi Master:


,mandi_id,mandi_name,district,state,mandi_type,total_area_acres
0,MANDI001,Hyderabad Mandi,Ludhiana,Punjab,Private,11.0
1,MANDI006,Kochi Mandi,Amritsar,Punjab,Direct,34.0
2,MANDI037,Jorhat Grain Market,Kurukshetra,Haryana,APMC,NaN
3,MANDI046,Baranagar Market,<NA>,Uttar Pradesh,Private,13.0
4,MANDI014,Bathinda Grain Market,Patiala,Punjab,Private,25.0
5,MANDI055,Chandigarh APMC,Bareilly,Uttar Pradesh,APMC,12.0
6,MANDI034,Ludhiana Grain Market,Ambala,Haryana,Unknown,11.0
7,MANDI049,Machilipatnam Mandi,Muzaffarnagar,Uttar Pradesh,Private,39.0
8,MANDI013,Aurangabad Market,Patiala,Punjab,Private,25.0
10,MANDI047,Orai Market,Muzaffarnagar,Uttar Pradesh,APMC,NaN



Mandi types:
mandi_type
APMC       20
Private    17
Unknown    11
Direct      9
Name: count, dtype: Int64

Districts:
['Agra', 'Ambala', 'Amritsar', 'Bareilly', 'Bathinda', 'Fatehabad', 'Ferozepur', 'Hisar', 'Jalandhar', 'Karnal', 'Kurukshetra', 'Ludhiana', 'Meerut', 'Moga', 'Muzaffarnagar', 'Patiala', 'Saharanpur', 'Sirsa']

Missing values:
mandi_id            0
mandi_name          0
district            4
state               4
mandi_type          0
total_area_acres    6
dtype: int64

Duplicate mandi IDs:
0


In [120]:
print(master_clean.shape)
print(master_clean["mandi_id"].nunique())
print(master_clean["mandi_type"].value_counts())
print(master_clean.isna().sum())

(57, 6)
57
mandi_type
APMC       20
Private    17
Unknown    11
Direct      9
Name: count, dtype: Int64
mandi_id            0
mandi_name          0
district            4
state               4
mandi_type          0
total_area_acres    6
dtype: int64


In [121]:
import os

# Define the target directory path
output_dir = "/content/cleaned"

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Now save your file safely
master_clean.to_csv(f"{output_dir}/mandi_master_clean.csv", index=False)


ARRIVAL DATA INSPECTION

In [122]:
# Basic information
print("Shape:", arrivals.shape)

print("\nColumns:")
print(arrivals.columns.tolist())

print("\nData types:")
print(arrivals.dtypes)

print("First 10 rows:")
display(arrivals.head(10))

print("Random 10 rows:")
display(arrivals.sample(10, random_state=42))

print("Missing values:")
missing = arrivals.isna().sum()

missing_percentage = (missing / len(arrivals) * 100).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": missing_percentage
})

display(missing_report)

print("Unique units:")
print(arrivals["unit"].value_counts(dropna=False))

print("\nUnique unit values:")
print(arrivals["unit"].unique())

print("Number of unique crop names:")
print(arrivals["crop_name"].nunique())

print("\nCrop names:")
print(arrivals["crop_name"].value_counts(dropna=False))

print("Sample date values:")
print(arrivals["date"].dropna().sample(20, random_state=42).tolist())

print("Sample arrival quantities:")
print(
    arrivals["arrival_quantity"]
    .dropna()
    .sample(20, random_state=42)
    .tolist()
)

print("Number of duplicate rows:", arrivals.duplicated().sum())

Shape: (25750, 8)

Columns:
['arrival_id', 'date', 'mandi_id', 'crop_name', 'variety', 'arrival_quantity', 'unit', 'farmer_count']

Data types:
arrival_id           object
date                 object
mandi_id             object
crop_name            object
variety              object
arrival_quantity     object
unit                 object
farmer_count        float64
dtype: object
First 10 rows:


,arrival_id,date,mandi_id,crop_name,variety,arrival_quantity,unit,farmer_count
0,ARR0012424,06-04-2026,MANDI026,गेहूं,Hybrid,44.841,T,128.0
1,ARR0008271,2026-06-02,MANDI014,Gehun,HD-2967,332.54,Qtl,33.0
2,ARR0022724,08-16-2026,056,Kanak,HD-2967,415.88 qtl,NaN,NaN
3,ARR0012573,06-24-2026,MANDI019,Maize,NaN,-107.13,KG,112.0
4,ARR0012098,09.08.2026,mandi050,Corn,Premium,-331.59,Qtl,NaN
5,ARR0013701,2026-03-31,mandi_049,Ganne,Local,119.93,Q,105.0
6,ARR0010753,10/07/2026,MANDI-054,Kapas,HD-2967,187.68,qtl,45.0
7,ARR0022123,15/07/2026,MANDI045,Sarso,Hybrid,393.67 qtl,NaN,24.0
8,ARR0003275,04-08-2026,mandi029,Sarson,PBW-343,14.84,Quintals,129.0
9,ARR0002542,2026-08-14,MANDI010,Mustard,Pusa-1121,93.28,qtl,109.0


Random 10 rows:


,arrival_id,date,mandi_id,crop_name,variety,arrival_quantity,unit,farmer_count
12423,ARR0009788,04.02.2026,MANDI004,चावल,Hybrid,35.722,Tonnes,74.0
8270,ARR0010773,03.03.2026,mandi_014,Ganne,Local,71.31,qtl,82.0
22723,ARR0007194,09/06/2026,MANDI043,Basmati,Hybrid,487.27,Quintals,114.0
12572,ARR0023253,21.01.2026,MANDI051,चावल,Local,253.91 qtl,NaN,93.0
12097,ARR0014932,14-Jun-2026,M018,Sarson,Pusa-1121,"12,270.0 KG",NaN,48.0
13700,ARR0001416,2026-05-24,MANDI-005,Paddy,Pusa-1121,475.35,qtl,NaN
10752,ARR0018689,03-26-2026,mandi_026,गेहूं,Hybrid,45.858999999999995,MT,NaN
22122,ARR0019504,21/06/2026,MANDI014,wheat,Hybrid,66.02 qtl,NaN,39.0
3274,ARR0022071,20/03/2026,M054,Rice,HD-2967,119.62 qtl,NaN,73.0
2541,ARR0005550,2026-05-20,MANDI033,Kapas,Local,11.433,Tonnes,24.0


Missing values:


,missing_count,missing_percentage
arrival_id,484,1.88
date,0,0.00
mandi_id,0,0.00
crop_name,0,0.00
variety,3735,14.50
arrival_quantity,0,0.00
unit,5143,19.97
farmer_count,3899,15.14


Unique units:
unit
NaN         5143
Qtl         2114
KG          1872
Q           1615
quintal     1571
qtl         1533
Quintals    1488
KGS         1341
Kilo        1338
MT          1331
T           1296
tonnes      1294
kg          1293
Kgs         1284
Tonnes      1237
Name: count, dtype: int64

Unique unit values:
['T' 'Qtl' nan 'KG' 'Q' 'qtl' 'Quintals' 'tonnes' 'kg' 'quintal' 'KGS'
 'MT' 'Kgs' 'Tonnes' 'Kilo']
Number of unique crop names:
36

Crop names:
crop_name
Sugarcane    918
Cotton       913
Mustard      909
गन्ना        896
Sarson       887
Sarso        874
कपास         863
sugarcane    851
सरसों        843
Narma        837
Ganna        831
mustard      816
cotton       812
Kapas        790
Ganne        782
Corn         749
मक्का        729
Makka        727
Maize        722
corn         705
wheat        668
गेहूं        661
Makki        645
Wheat        644
Gehun        638
Kanak        624
WHEAT        609
GEHUN        587
Chawal       575
paddy        551
चावल         5

In [123]:
# ==========================================
# 5.2 Detailed Arrivals Inspection
# ==========================================

# 1. Check all unique crop names
print("ALL CROP NAMES:")
print(sorted(arrivals["crop_name"].astype(str).unique()))


# 2. Check all unique units
print("\nALL UNITS:")
print(arrivals["unit"].unique())


# 3. Check mandi ID formats
print("\nSAMPLE MANDI IDs:")
print(arrivals["mandi_id"].unique()[:50])


# 4. Check arrival quantity as raw strings
print("\nSAMPLE RAW QUANTITIES:")
display(
    arrivals[["arrival_quantity", "unit"]]
    .drop_duplicates()
    .sample(30, random_state=42)
)


# 5. Check whether quantities contain letters/units
print("\nQUANTITIES CONTAINING TEXT:")
quantity_text = arrivals[
    arrivals["arrival_quantity"].astype(str).str.contains(
        r"[A-Za-z]",
        regex=True,
        na=False
    )
]

display(
    quantity_text[["arrival_quantity", "unit"]]
    .head(30)
)


# 6. Check negative quantities
print("\nNEGATIVE QUANTITIES:")

# Extract numeric portion for inspection only
temp_quantity = (
    arrivals["arrival_quantity"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"([-+]?\d*\.?\d+)")[0]
)

temp_quantity = pd.to_numeric(temp_quantity, errors="coerce")

print("Negative quantity count:", (temp_quantity < 0).sum())

display(
    arrivals.loc[
        temp_quantity < 0,
        ["date", "mandi_id", "crop_name", "arrival_quantity", "unit"]
    ].head(20)
)

ALL CROP NAMES:
['Basmati', 'Chawal', 'Corn', 'Cotton', 'Dhaan', 'GEHUN', 'Ganna', 'Ganne', 'Gehun', 'Kanak', 'Kapas', 'Maize', 'Makka', 'Makki', 'Mustard', 'Narma', 'Paddy', 'Rice', 'Sarso', 'Sarson', 'Sugarcane', 'WHEAT', 'Wheat', 'corn', 'cotton', 'mustard', 'paddy', 'sugarcane', 'wheat', 'कपास', 'गन्ना', 'गेहूं', 'चावल', 'धान', 'मक्का', 'सरसों']

ALL UNITS:
['T' 'Qtl' nan 'KG' 'Q' 'qtl' 'Quintals' 'tonnes' 'kg' 'quintal' 'KGS'
 'MT' 'Kgs' 'Tonnes' 'Kilo']

SAMPLE MANDI IDs:
['MANDI026' 'MANDI014' '056' 'MANDI019' 'mandi050' 'mandi_049' 'MANDI-054'
 'MANDI045' 'mandi029' 'MANDI010' 'mandi_044' 'MANDI042' 'MANDI-056'
 'MANDI011' '025' 'MANDI050' 'MANDI006' 'MANDI039' 'MANDI-001' 'MANDI028'
 'mandi_045' 'MANDI-027' 'MANDI024' 'MANDI022' 'MANDI044' 'M045' '021'
 'MANDI-012' 'MANDI055' 'mandi_016' 'mandi_001' 'MANDI037' 'MANDI008'
 'MANDI047' 'M051' 'MANDI-023' 'MANDI017' 'mandi_002' 'MANDI016'
 'MANDI-018' 'MANDI054' 'mandi013' 'MANDI007' '057' 'mandi_025'
 'MANDI-041' 'MANDI040' 'mand

,arrival_quantity,unit
970,3817.0,KGS
1122,347.03,Qtl
8223,10519.0,KG
25137,13268.0,Kilo
1486,-17.99,KG
12490,335.73,qtl
19543,376.42,qtl
15520,36.147000000000006,T
10745,38.12 qtl,NaN
5663,330.9 qtl,NaN



QUANTITIES CONTAINING TEXT:


,arrival_quantity,unit
2,415.88 qtl,NaN
7,393.67 qtl,NaN
34,"36,654.0 KG",NaN
40,"22,697.0 KG",NaN
44,336.73 qtl,NaN
52,309.07 qtl,NaN
57,431.25 qtl,NaN
65,"43,655.0 KG",NaN
68,"7,278.0 KG",NaN
78,76.9 qtl,NaN



NEGATIVE QUANTITIES:
Negative quantity count: 1261


,date,mandi_id,crop_name,arrival_quantity,unit
3,06-24-2026,MANDI019,Maize,-107.13,KG
4,09.08.2026,mandi050,Corn,-331.59,Qtl
48,06/05/2026,MANDI007,sugarcane,-177.05,KG
92,2026/01/15,mandi_054,Wheat,-357.18,KG
130,2026/01/09,MANDI039,Dhaan,-54.89,KG
133,27-Mar-2026,MANDI023,wheat,-455.48,KG
135,03-Feb-2026,MANDI-036,कपास,-453.42,KG
138,04.04.2026,MANDI046,Dhaan,-77.12,Qtl
153,18/08/2026,MANDI-049,Mustard,-11.34,KG
165,2026-07-10,mandi_024,Maize,-77.75,Qtl


In [124]:
# Never modify the raw dataset directly
arrivals_clean = arrivals.copy()

print("Raw rows:", len(arrivals_clean))

# Remove exact duplicate records
before = len(arrivals_clean)

arrivals_clean = arrivals_clean.drop_duplicates()

after = len(arrivals_clean)

print("Duplicates removed:", before - after)
print("Rows remaining:", after)

# Standardize existing IDs
arrivals_clean["arrival_id"] = (
    arrivals_clean["arrival_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Generate IDs for missing arrival IDs
missing_id = arrivals_clean["arrival_id"].isna()

arrivals_clean.loc[missing_id, "arrival_id"] = [
    f"ARR_MISSING_{i:05d}"
    for i in range(1, missing_id.sum() + 1)
]

print("Missing arrival IDs after cleaning:",
      arrivals_clean["arrival_id"].isna().sum())

def clean_mandi_id(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().upper()

    # Remove separators
    x = x.replace("-", "").replace("_", "").replace(" ", "")

    # M056 -> MANDI056
    if x.startswith("M") and not x.startswith("MANDI"):
        number = x[1:]

        if number.isdigit():
            return f"MANDI{int(number):03d}"

    # 056 -> MANDI056
    if x.isdigit():
        return f"MANDI{int(x):03d}"

    # MANDI56 -> MANDI056
    if x.startswith("MANDI"):
        number = x[5:]

        if number.isdigit():
            return f"MANDI{int(number):03d}"

    return x


arrivals_clean["mandi_id"] = (
    arrivals_clean["mandi_id"]
    .apply(clean_mandi_id)
)

print(arrivals_clean["mandi_id"].unique()[:20])

crop_mapping = {

    # Wheat
    "Wheat": "Wheat",
    "wheat": "Wheat",
    "WHEAT": "Wheat",
    "GEHUN": "Wheat",
    "Gehun": "Wheat",
    "Kanak": "Wheat",
    "गेहूं": "Wheat",

    # Rice
    "Rice": "Rice",
    "Paddy": "Rice",
    "paddy": "Rice",
    "Dhaan": "Rice",
    "Chawal": "Rice",
    "चावल": "Rice",
    "धान": "Rice",

    # Maize
    "Maize": "Maize",
    "Corn": "Maize",
    "corn": "Maize",
    "Makka": "Maize",
    "Makki": "Maize",
    "मक्का": "Maize",

    # Mustard
    "Mustard": "Mustard",
    "mustard": "Mustard",
    "Sarso": "Mustard",
    "Sarson": "Mustard",
    "सरसों": "Mustard",

    # Cotton
    "Cotton": "Cotton",
    "cotton": "Cotton",
    "Kapas": "Cotton",
    "Narma": "Cotton",
    "कपास": "Cotton",

    # Sugarcane
    "Sugarcane": "Sugarcane",
    "sugarcane": "Sugarcane",
    "Ganna": "Sugarcane",
    "Ganne": "Sugarcane",
    "गन्ना": "Sugarcane",

    # Basmati
    "Basmati": "Basmati"
}

arrivals_clean["crop_name"] = (
    arrivals_clean["crop_name"]
    .astype("string")
    .str.strip()
    .map(crop_mapping)
)

print(arrivals_clean["crop_name"].value_counts(dropna=False))


quantity_raw = (
    arrivals_clean["arrival_quantity"]
    .astype("string")
    .str.strip()
)

embedded_unit = quantity_raw.str.extract(
    r"(?i)\b(qtl|quintals?|kg|kgs|kilo|tonnes?|mt|t)\b",
    expand=False
)

print(embedded_unit.value_counts(dropna=False))

unit_missing = arrivals_clean["unit"].isna()

arrivals_clean.loc[unit_missing, "unit"] = (
    embedded_unit[unit_missing]
)

print("Missing units after recovery:",
      arrivals_clean["unit"].isna().sum())

unit_mapping = {

    # Kilograms
    "kg": "KG",
    "KG": "KG",
    "KGS": "KG",
    "Kgs": "KG",
    "kilo": "KG",

    # Quintals
    "Q": "QTL",
    "qtl": "QTL",
    "Qtl": "QTL",
    "quintal": "QTL",
    "Quintals": "QTL",

    # Tonnes
    "T": "TONNE",
    "MT": "TONNE",
    "tonnes": "TONNE",
    "Tonnes": "TONNE"
}

arrivals_clean["unit"] = arrivals_clean["unit"].map(unit_mapping)

print(arrivals_clean["unit"].value_counts(dropna=False))

arrivals_clean["quantity_numeric"] = (
    arrivals_clean["arrival_quantity"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.extract(r"([-+]?\d*\.?\d+)", expand=False)
)

arrivals_clean["quantity_numeric"] = pd.to_numeric(
    arrivals_clean["quantity_numeric"],
    errors="coerce"
)

print(arrivals_clean["quantity_numeric"].describe())

arrivals_clean["arrival_quantity_qtl"] = np.nan

# KG → Quintals
kg_mask = arrivals_clean["unit"] == "KG"

arrivals_clean.loc[kg_mask, "arrival_quantity_qtl"] = (
    arrivals_clean.loc[kg_mask, "quantity_numeric"] / 100
)

# Quintal → Quintals
qtl_mask = arrivals_clean["unit"] == "QTL"

arrivals_clean.loc[qtl_mask, "arrival_quantity_qtl"] = (
    arrivals_clean.loc[qtl_mask, "quantity_numeric"]
)

# Tonne → Quintals
tonne_mask = arrivals_clean["unit"] == "TONNE"

arrivals_clean.loc[tonne_mask, "arrival_quantity_qtl"] = (
    arrivals_clean.loc[tonne_mask, "quantity_numeric"] * 10
)

print(
    arrivals_clean[
        ["arrival_quantity", "unit", "quantity_numeric",
         "arrival_quantity_qtl"]
    ].head(20)
)

negative_mask = arrivals_clean["arrival_quantity_qtl"] < 0

negative_count = negative_mask.sum()

print("Negative records:", negative_count)

# Remove invalid negative arrivals
arrivals_clean = arrivals_clean.loc[
    ~negative_mask
].copy()

print("Rows after removing negative quantities:",
      len(arrivals_clean))

arrivals_clean["date"] = pd.to_datetime(
    arrivals_clean["date"],
    errors="coerce",
    dayfirst=False
)

print("Invalid dates:",
      arrivals_clean["date"].isna().sum())

print(
    arrivals_clean["date"].min(),
    "to",
    arrivals_clean["date"].max()
)



Raw rows: 25750
Duplicates removed: 750
Rows remaining: 25000
Missing arrival IDs after cleaning: 0
['MANDI026' 'MANDI014' 'MANDI056' 'MANDI019' 'MANDI050' 'MANDI049'
 'MANDI054' 'MANDI045' 'MANDI029' 'MANDI010' 'MANDI044' 'MANDI042'
 'MANDI011' 'MANDI025' 'MANDI006' 'MANDI039' 'MANDI001' 'MANDI028'
 'MANDI027' 'MANDI024']
crop_name
Wheat        4284
Mustard      4209
Sugarcane    4161
Maize        4155
Cotton       4110
Rice         3561
Basmati       520
Name: count, dtype: int64
arrival_quantity
<NA>    19996
KG       2519
qtl      2485
Name: count, dtype: Int64
Missing units after recovery: 0
unit
QTL      10561
KG        8126
TONNE     5006
NaN       1307
Name: count, dtype: int64
count         25000.0
mean      9110.349275
std      14787.536167
min            -499.6
25%          43.17375
50%           315.995
75%          15241.75
max           49998.0
Name: quantity_numeric, dtype: Float64
      arrival_quantity   unit  quantity_numeric  arrival_quantity_qtl
0               44.8

In [125]:
print("===================================")
print("ARRIVALS CLEANING SUMMARY")
print("===================================")

print("Rows:", len(arrivals_clean))
print("Columns:", arrivals_clean.shape[1])

print("\nMissing values:")
print(arrivals_clean.isna().sum())

print("\nUnits:")
print(arrivals_clean["unit"].value_counts(dropna=False))

print("\nCrops:")
print(arrivals_clean["crop_name"].value_counts())

print("\nDuplicate rows:",
      arrivals_clean.duplicated().sum())

print("\nNegative quantities:",
      (arrivals_clean["arrival_quantity_qtl"] < 0).sum())

print("\nMandi IDs not matching MANDI###:")
print(
    arrivals_clean[
        ~arrivals_clean["mandi_id"].str.match(
            r"^MANDI\d{3}$",
            na=False
        )
    ]["mandi_id"].unique()
)

ARRIVALS CLEANING SUMMARY
Rows: 23767
Columns: 10

Missing values:
arrival_id                  0
date                    20260
mandi_id                    0
crop_name                   0
variety                  3448
arrival_quantity            0
unit                     1307
farmer_count             3631
quantity_numeric            0
arrival_quantity_qtl     1307
dtype: int64

Units:
unit
QTL      9950
KG       7504
TONNE    5006
NaN      1307
Name: count, dtype: int64

Crops:
crop_name
Wheat        4074
Mustard      4024
Sugarcane    3960
Maize        3955
Cotton       3870
Rice         3387
Basmati       497
Name: count, dtype: int64

Duplicate rows: 0

Negative quantities: 0

Mandi IDs not matching MANDI###:
[]


In [126]:
# ==========================================
# FIX UNIT STANDARDIZATION
# ==========================================

# Start from the current arrivals_clean
# Normalize unit text first

arrivals_clean["unit"] = (
    arrivals_clean["unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

unit_mapping = {
    # KG
    "kg": "KG",
    "kgs": "KG",
    "kilo": "KG",

    # Quintals
    "q": "QTL",
    "qtl": "QTL",
    "quintal": "QTL",
    "quintals": "QTL",

    # Tonnes
    "t": "TONNE",
    "mt": "TONNE",
    "tonne": "TONNE",
    "tonnes": "TONNE"
}

arrivals_clean["unit"] = arrivals_clean["unit"].map(unit_mapping)

print("Units after standardization:")
print(arrivals_clean["unit"].value_counts(dropna=False))

print("\nRows still missing unit:")
display(
    arrivals_clean[
        arrivals_clean["unit"].isna()
    ][["arrival_quantity", "unit"]].head(30)
)

# ==========================================
# DATE FORMAT INSPECTION
# ==========================================

date_text = (
    arrivals_clean["date"]
    .astype("string")
    .str.strip()
)

# Show examples of different formats
print("Sample date values:")
print(date_text.drop_duplicates().sample(50, random_state=42).tolist())

print("\nNumber of unique date strings:")
print(date_text.nunique())

print("\nExamples by separator:")

print("\nHyphen:")
print(
    date_text[
        date_text.str.contains("-", na=False)
    ].drop_duplicates().head(20).tolist()
)

print("\nSlash:")
print(
    date_text[
        date_text.str.contains("/", na=False)
    ].drop_duplicates().head(20).tolist()
)

print("\nDot:")
print(
    date_text[
        date_text.str.contains(".", regex=False, na=False)
    ].drop_duplicates().head(20).tolist()
)

print("\nDate parsing test:")

test_dates = [
    "06-04-2026",
    "08-16-2026",
    "2026-06-02",
    "09.08.2026",
    "10/07/2026",
    "14-Jun-2026",
    "03-26-2026"
]

for d in test_dates:
    print(d)

Units after standardization:
unit
QTL      9950
KG       7504
TONNE    5006
NaN      1307
Name: count, dtype: int64

Rows still missing unit:


,arrival_quantity,unit
29,35053.0,NaN
39,15999.0,NaN
45,12790.0,NaN
60,27041.000000000004,NaN
74,5918.0,NaN
125,6202.0,NaN
126,16784.0,NaN
143,8974.0,NaN
144,32314.999999999996,NaN
177,44371.0,NaN


Sample date values:
['2026-05-05', '2026-08-08', '2026-02-24', '2026-09-04', '2026-06-14', '2026-03-11', '2026-05-19', '2026-06-28', '2026-05-15', '2026-01-24', '2026-09-09', '2026-03-09', '2026-08-23', '2026-07-09', '2026-04-20', '2026-06-15', '2026-08-18', '2026-07-25', '2026-09-01', '2026-06-02', '2026-01-11', '2026-04-01', '2026-03-02', '2026-03-31', '2026-07-10', '2026-06-18', '2026-05-01', '2026-05-21', '2026-05-20', '2026-07-21', '2026-05-13', '2026-09-08', '2026-01-03', '2026-01-18', '2026-06-11', '2026-07-19', '2026-02-22', '2026-06-05', '2026-09-07', '2026-05-02', '2026-04-05', '2026-01-10', '2026-06-29', '2026-07-17', '2026-04-23', '2026-04-12', '2026-02-19', '2026-02-16', '2026-05-07', '2026-08-21']

Number of unique date strings:
252

Examples by separator:

Hyphen:
['2026-06-04', '2026-08-16', '2026-04-08', '2026-03-20', '2026-06-03', '2026-08-08', '2026-06-08', '2026-08-05', '2026-05-15', '2026-05-20', '2026-04-04', '2026-06-25', '2026-08-25', '2026-03-28', '2026-07-25',

In [127]:
# ==========================================
# INVESTIGATE MISSING UNITS
# ==========================================

missing_unit = arrivals_clean["unit"].isna()

print("Rows with missing unit:", missing_unit.sum())

print("\nMissing units by crop:")
display(
    arrivals_clean.loc[missing_unit, "crop_name"]
    .value_counts()
)

print("\nMissing units by mandi:")
display(
    arrivals_clean.loc[missing_unit, "mandi_id"]
    .value_counts()
)

print("\nMissing units by date:")
display(
    arrivals_clean.loc[missing_unit, "date"]
    .value_counts()
    .head(20)
)

print("\nMissing-unit quantity statistics:")
display(
    arrivals_clean.loc[
        missing_unit,
        "quantity_numeric"
    ].describe()
)

# ==========================================
# ROBUST DATE CLEANING
# ==========================================

# Restore original date strings
arrivals_clean["date_raw"] = (
    arrivals.loc[arrivals_clean.index, "date"]
    .astype("string")
    .str.strip()
)

date_raw = arrivals_clean["date_raw"]

# Start with empty datetime column
arrivals_clean["date"] = pd.NaT

# ------------------------------------------
# Format 1: YYYY-MM-DD
# ------------------------------------------

mask_iso = date_raw.str.match(
    r"^\d{4}-\d{2}-\d{2}$",
    na=False
)

arrivals_clean.loc[mask_iso, "date"] = pd.to_datetime(
    date_raw[mask_iso],
    format="%Y-%m-%d",
    errors="coerce"
)


# ------------------------------------------
# Format 2: MM-DD-YYYY
# ------------------------------------------

mask_mdy_dash = date_raw.str.match(
    r"^\d{2}-\d{2}-\d{4}$",
    na=False
)

arrivals_clean.loc[mask_mdy_dash, "date"] = pd.to_datetime(
    date_raw[mask_mdy_dash],
    format="%m-%d-%Y",
    errors="coerce"
)


# ------------------------------------------
# Format 3: MM/DD/YYYY
# ------------------------------------------

mask_mdy_slash = date_raw.str.match(
    r"^\d{2}/\d{2}/\d{4}$",
    na=False
)

arrivals_clean.loc[mask_mdy_slash, "date"] = pd.to_datetime(
    date_raw[mask_mdy_slash],
    format="%m/%d/%Y",
    errors="coerce"
)


# ------------------------------------------
# Format 4: MM.DD.YYYY
# ------------------------------------------

mask_mdy_dot = date_raw.str.match(
    r"^\d{2}\.\d{2}\.\d{4}$",
    na=False
)

arrivals_clean.loc[mask_mdy_dot, "date"] = pd.to_datetime(
    date_raw[mask_mdy_dot],
    format="%m.%d.%Y",
    errors="coerce"
)


# ------------------------------------------
# Format 5: DD-Mon-YYYY
# Example: 14-Jun-2026
# ------------------------------------------

mask_dmony = date_raw.str.match(
    r"^\d{1,2}-[A-Za-z]{3}-\d{4}$",
    na=False
)

arrivals_clean.loc[mask_dmony, "date"] = pd.to_datetime(
    date_raw[mask_dmony],
    format="%d-%b-%Y",
    errors="coerce"
)


# ------------------------------------------
# Validation
# ------------------------------------------

print("Total rows:", len(arrivals_clean))
print("Valid dates:", arrivals_clean["date"].notna().sum())
print("Invalid dates:", arrivals_clean["date"].isna().sum())

print("\nDate range:")
print(arrivals_clean["date"].min())
print(arrivals_clean["date"].max())

invalid_dates = arrivals_clean[
    arrivals_clean["date"].isna()
]

print("Invalid date rows:", len(invalid_dates))

display(
    invalid_dates[
        ["date_raw", "mandi_id", "crop_name", "arrival_quantity"]
    ].head(50)
)

Rows with missing unit: 1307

Missing units by crop:


,count
crop_name,
Wheat,230
Maize,227
Cotton,219
Sugarcane,214
Mustard,206
Rice,187
Basmati,24



Missing units by mandi:


,count
mandi_id,
MANDI022,35
MANDI039,32
MANDI038,32
MANDI041,32
MANDI003,30
MANDI006,29
MANDI055,28
MANDI028,28
MANDI053,28



Missing units by date:


,count
date,
2026-06-07,4
2026-07-20,4
2026-03-20,4
2026-09-04,3
2026-09-01,3
2026-07-25,3
2026-06-19,3
2026-08-18,3
2026-04-27,2



Missing-unit quantity statistics:


,quantity_numeric
count,1307.0
mean,25983.690895
std,14192.524076
min,1023.0
25%,13979.5
50%,26655.0
75%,38105.0
max,49998.0


Total rows: 23767
Valid dates: 15317
Invalid dates: 8450

Date range:
2026-01-01 00:00:00
2026-12-08 00:00:00
Invalid date rows: 8450


,date_raw,mandi_id,crop_name,arrival_quantity
7,15/07/2026,MANDI045,Mustard,393.67 qtl
12,31/03/2026,MANDI056,Maize,37.3
13,16/01/2026,MANDI011,Sugarcane,44.83
14,27.06.2026,MANDI025,Wheat,45.474000000000004
15,18.01.2026,MANDI050,Sugarcane,1518.0
17,2026/01/09,MANDI039,Wheat,41.577
22,18.06.2026,MANDI026,Maize,264.38
24,30.04.2026,MANDI024,Mustard,7048.999999999999
25,21/08/2026,MANDI022,Maize,38.343
26,2026/05/31,MANDI044,Rice,113.89


In [128]:
# ==========================================
# ROBUST DATE PARSER - FINAL VERSION
# ==========================================

# Always go back to the original raw date column
arrivals_clean["date_raw"] = (
    arrivals.loc[arrivals_clean.index, "date"]
    .astype("string")
    .str.strip()
)

date_raw = arrivals_clean["date_raw"]

# Start empty
arrivals_clean["date"] = pd.NaT


# --------------------------------------------------
# 1. YYYY-MM-DD
# Example: 2026-06-04
# --------------------------------------------------

mask = date_raw.str.match(
    r"^\d{4}-\d{2}-\d{2}$",
    na=False
)

arrivals_clean.loc[mask, "date"] = pd.to_datetime(
    date_raw[mask],
    format="%Y-%m-%d",
    errors="coerce"
)


# --------------------------------------------------
# 2. MM-DD-YYYY
# Examples: 08-16-2026, 03-26-2026
#
# These are clearly month-first when the second
# number is > 12.
# --------------------------------------------------

mask = date_raw.str.match(
    r"^\d{2}-\d{2}-\d{4}$",
    na=False
)

arrivals_clean.loc[mask, "date"] = pd.to_datetime(
    date_raw[mask],
    format="%m-%d-%Y",
    errors="coerce"
)


# --------------------------------------------------
# 3. DD/MM/YYYY
# Example: 15/07/2026
# --------------------------------------------------

mask = date_raw.str.match(
    r"^\d{2}/\d{2}/\d{4}$",
    na=False
)

arrivals_clean.loc[mask, "date"] = pd.to_datetime(
    date_raw[mask],
    format="%d/%m/%Y",
    errors="coerce"
)


# --------------------------------------------------
# 4. DD.MM.YYYY
# Example: 27.06.2026
# --------------------------------------------------

mask = date_raw.str.match(
    r"^\d{2}\.\d{2}\.\d{4}$",
    na=False
)

arrivals_clean.loc[mask, "date"] = pd.to_datetime(
    date_raw[mask],
    format="%d.%m.%Y",
    errors="coerce"
)


# --------------------------------------------------
# 5. YYYY/MM/DD
# Example: 2026/07/15
# --------------------------------------------------

mask = date_raw.str.match(
    r"^\d{4}/\d{2}/\d{2}$",
    na=False
)

arrivals_clean.loc[mask, "date"] = pd.to_datetime(
    date_raw[mask],
    format="%Y/%m/%d",
    errors="coerce"
)


# --------------------------------------------------
# 6. DD-Mon-YYYY
# Example: 14-Jun-2026
# --------------------------------------------------

mask = date_raw.str.match(
    r"^\d{1,2}-[A-Za-z]{3}-\d{4}$",
    na=False
)

arrivals_clean.loc[mask, "date"] = pd.to_datetime(
    date_raw[mask],
    format="%d-%b-%Y",
    errors="coerce"
)


# --------------------------------------------------
# Validation
# --------------------------------------------------

print("Total rows:", len(arrivals_clean))
print("Valid dates:", arrivals_clean["date"].notna().sum())
print("Invalid dates:", arrivals_clean["date"].isna().sum())

print("\nDate range:")
print("Minimum:", arrivals_clean["date"].min())
print("Maximum:", arrivals_clean["date"].max())

# Show any dates that still failed

invalid_dates = arrivals_clean[
    arrivals_clean["date"].isna()
]

print("Invalid date rows:", len(invalid_dates))

if len(invalid_dates) > 0:
    display(
        invalid_dates[
            ["date_raw", "mandi_id", "crop_name", "arrival_quantity"]
        ].head(50)
    )

    print("\nDates outside expected 2026 range:")

display(
    arrivals_clean[
        (arrivals_clean["date"] < "2026-01-01") |
        (arrivals_clean["date"] > "2026-12-31")
    ][["date_raw", "date"]].head(20)
)

# ==========================================
# HANDLE UNKNOWN UNITS
# ==========================================

# Any unit that couldn't be recovered is UNKNOWN
arrivals_clean["unit"] = (
    arrivals_clean["unit"]
    .fillna("UNKNOWN")
)

print(arrivals_clean["unit"].value_counts())

# ==========================================
# FINAL QUANTITY → QUINTALS CONVERSION
# ==========================================

arrivals_clean["arrival_quantity_qtl"] = np.nan

# KG → QTL
mask = arrivals_clean["unit"] == "KG"

arrivals_clean.loc[mask, "arrival_quantity_qtl"] = (
    arrivals_clean.loc[mask, "quantity_numeric"] / 100
)

# QTL → QTL
mask = arrivals_clean["unit"] == "QTL"

arrivals_clean.loc[mask, "arrival_quantity_qtl"] = (
    arrivals_clean.loc[mask, "quantity_numeric"]
)

# TONNE → QTL
mask = arrivals_clean["unit"] == "TONNE"

arrivals_clean.loc[mask, "arrival_quantity_qtl"] = (
    arrivals_clean.loc[mask, "quantity_numeric"] * 10
)

print(
    "Unknown-unit records:",
    (arrivals_clean["unit"] == "UNKNOWN").sum()
)

print(
    "Missing converted quantities:",
    arrivals_clean["arrival_quantity_qtl"].isna().sum()
)

arrivals_clean["quantity_quality"] = "VALID"

arrivals_clean.loc[
    arrivals_clean["unit"] == "UNKNOWN",
    "quantity_quality"
] = "UNKNOWN_UNIT"

arrivals_clean["quantity_quality"].value_counts()

Total rows: 23767
Valid dates: 23767
Invalid dates: 0

Date range:
Minimum: 2026-01-01 00:00:00
Maximum: 2026-09-09 00:00:00
Invalid date rows: 0


,date_raw,date


unit
QTL        9950
KG         7504
TONNE      5006
UNKNOWN    1307
Name: count, dtype: int64
Unknown-unit records: 1307
Missing converted quantities: 1307


,count
quantity_quality,
VALID,22460
UNKNOWN_UNIT,1307


In [129]:
# ==========================================
# FINAL ARRIVALS VALIDATION
# ==========================================

print("===================================")
print("FINAL ARRIVALS VALIDATION")
print("===================================")

print("\nRows:", len(arrivals_clean))

print("\nMissing values:")
print(arrivals_clean.isna().sum())

print("\nUnits:")
print(arrivals_clean["unit"].value_counts())

print("\nQuantity quality:")
print(arrivals_clean["quantity_quality"].value_counts())

print("\nNegative QTL:")
print(
    (arrivals_clean["arrival_quantity_qtl"] < 0).sum()
)

print("\nDuplicate rows:")
print(arrivals_clean.duplicated().sum())

print("\nInvalid dates:")
print(arrivals_clean["date"].isna().sum())

print("\nDate range:")
print(
    arrivals_clean["date"].min(),
    "→",
    arrivals_clean["date"].max()
)

print("\nInvalid Mandi IDs:")
print(
    (~arrivals_clean["mandi_id"].str.match(
        r"^MANDI\d{3}$",
        na=False
    )).sum()
)

FINAL ARRIVALS VALIDATION

Rows: 23767

Missing values:
arrival_id                 0
date                       0
mandi_id                   0
crop_name                  0
variety                 3448
arrival_quantity           0
unit                       0
farmer_count            3631
quantity_numeric           0
arrival_quantity_qtl    1307
date_raw                   0
quantity_quality           0
dtype: int64

Units:
unit
QTL        9950
KG         7504
TONNE      5006
UNKNOWN    1307
Name: count, dtype: int64

Quantity quality:
quantity_quality
VALID           22460
UNKNOWN_UNIT     1307
Name: count, dtype: int64

Negative QTL:
0

Duplicate rows:
0

Invalid dates:
0

Date range:
2026-01-01 00:00:00 → 2026-09-09 00:00:00

Invalid Mandi IDs:
0


In [130]:
# ==========================================
# ARRIVALS → MANDI MASTER VALIDATION
# ==========================================

# Get unique mandi IDs from both datasets
arrival_mandis = set(arrivals_clean["mandi_id"].dropna().unique())
master_mandis = set(master_clean["mandi_id"].dropna().unique())

# IDs appearing in Arrivals but NOT in Master
missing_from_master = arrival_mandis - master_mandis

# IDs in Master that have no Arrivals
master_without_arrivals = master_mandis - arrival_mandis

print("Unique mandi IDs in Arrivals:", len(arrival_mandis))
print("Unique mandi IDs in Master:", len(master_mandis))

print("\nArrivals mandi IDs missing from Master:")
print(sorted(missing_from_master))

print("\nMaster mandi IDs with no Arrivals:")
print(sorted(master_without_arrivals))

# Count actual arrival rows whose mandi is not in master
invalid_mandi_rows = arrivals_clean[
    ~arrivals_clean["mandi_id"].isin(master_mandis)
]

print(
    "\nArrival rows with mandi IDs missing from Master:",
    len(invalid_mandi_rows)
)

Unique mandi IDs in Arrivals: 57
Unique mandi IDs in Master: 57

Arrivals mandi IDs missing from Master:
[]

Master mandi IDs with no Arrivals:
[]

Arrival rows with mandi IDs missing from Master: 0


In [131]:
# Remove temporary audit column
arrivals_clean = arrivals_clean.drop(columns=["date_raw"], errors="ignore")

# Save cleaned Arrivals
arrivals_clean.to_csv(
    "/content/cleaned/arrivals_clean.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [132]:
# Price & MSP - Raw Data Inspection

print("Shape:", price.shape)

print("\nColumns:")
print(price.columns.tolist())

print("\nData types:")
print(price.dtypes)

print("\nFirst 5 rows:")
display(price.head())

print("\nMissing values:")
print(price.isna().sum())

print("\nDuplicate rows:", price.duplicated().sum())

print("\nUnique values per column:")
for col in price.columns:
    print(f"\n--- {col} ---")
    print(price[col].dropna().astype(str).unique()[:20])

Shape: (12000, 9)

Columns:
['record_id', 'date', 'mandi_id', 'district', 'crop_name', 'min_price', 'max_price', 'modal_price', 'msp']

Data types:
record_id      object
date           object
mandi_id       object
district       object
crop_name      object
min_price      object
max_price      object
modal_price    object
msp            object
dtype: object

First 5 rows:


,record_id,date,mandi_id,district,crop_name,min_price,max_price,modal_price,msp
0,PR001650,2026/07/26,MANDI005,None,Kapas,6850.985817820893,"₹7,570.17",7210.58,"₹6,620.00"
1,PR000246,2026-07-17,MANDI028,Fatehabad,कपास,"₹6,944.79",7654.18,"Rs. 7,299",
2,PR010091,09.01.2026,MANDI011,Jalandhar,Dhaan,"Rs. 1,857",2013.18,"₹1,935.06","INR 2,183"
3,PR000982,2026/05/24,M012,Ferozepur,corn,"₹1,873.34","Rs. 1,986","Rs. 1,930","Rs. 2,090"
4,PR001708,18/08/2026,013,Karnal,Cotton,"5,878.62/-",6574.283051769212,6226.45,"Rs. 6,620"



Missing values:
record_id         0
date              0
mandi_id       1235
district        773
crop_name         0
min_price         0
max_price         0
modal_price       0
msp               0
dtype: int64

Duplicate rows: 0

Unique values per column:

--- record_id ---
['PR001650' 'PR000246' 'PR010091' 'PR000982' 'PR001708' 'PR004312'
 'PR000288' 'PR002536' 'PR009427' 'PR003418' 'PR001250' 'PR008959'
 'PR004344' 'PR008040' 'PR002773' 'PR007582' 'PR001390' 'PR011220'
 'PR008656' 'PR000719']

--- date ---
['2026/07/26' '2026-07-17' '09.01.2026' '2026/05/24' '18/08/2026'
 '01-24-2026' '08-Aug-2026' '2026-01-06' '12-Aug-2026' '09.02.2026'
 '10/02/2026' '24.06.2026' '2026-04-14' '25-Jun-2026' '06/04/2026'
 '24.01.2026' '06-17-2026' '17/01/2026' '26/03/2026' '2026-04-29']

--- mandi_id ---
['MANDI005' 'MANDI028' 'MANDI011' 'M012' '013' 'MANDI001' 'M031' 'M036'
 'MANDI-050' 'M011' 'MANDI002' 'mandi_057' 'mandi_040' 'MANDI053'
 'mandi_050' 'M043' 'MANDI026' 'MANDI-010' 'mandi005' 'MANDI01

In [133]:
# Inspect fields that are likely to need cleaning

print("Mandi ID examples:")
print(price["mandi_id"].dropna().astype(str).unique()[:30])

print("\nDistrict examples:")
print(price["district"].dropna().astype(str).unique()[:30])

print("\nCrop name examples:")
print(price["crop_name"].dropna().astype(str).unique()[:30])

print("\nMin price examples:")
print(price["min_price"].dropna().astype(str).unique()[:30])

print("\nMax price examples:")
print(price["max_price"].dropna().astype(str).unique()[:30])

print("\nModal price examples:")
print(price["modal_price"].dropna().astype(str).unique()[:30])

print("\nMSP examples:")
print(price["msp"].dropna().astype(str).unique()[:30])

print("\nDate examples:")
print(price["date"].dropna().astype(str).unique()[:30])

Mandi ID examples:
['MANDI005' 'MANDI028' 'MANDI011' 'M012' '013' 'MANDI001' 'M031' 'M036'
 'MANDI-050' 'M011' 'MANDI002' 'mandi_057' 'mandi_040' 'MANDI053'
 'mandi_050' 'M043' 'MANDI026' 'MANDI-010' 'mandi005' 'MANDI016'
 'MANDI024' '026' 'MANDI007' 'mandi_043' 'MANDI008' 'MANDI-027' 'MANDI030'
 'MANDI-056' 'mandi_039' 'M024']

District examples:
['Fatehabad' 'Jalandhar' 'Ferozepur' 'Karnal' 'Patiala' 'Hisar'
 'Kurukshetra' '' 'Ambala' 'Amritsar' 'Ludhiana' 'Bathinda' 'Sirsa' 'Moga']

Crop name examples:
['Kapas' 'कपास' 'Dhaan' 'corn' 'Cotton' 'Kanak' 'WHEAT' 'Gehun' 'गन्ना'
 'सरसों' 'Ganna' 'Sugarcane' 'Basmati' 'Makka' 'cotton' 'sugarcane'
 'mustard' 'wheat' 'Mustard' 'Corn' 'Ganne' 'Maize' 'धान' 'चावल' 'GEHUN'
 'paddy' 'Narma' 'गेहूं' 'Paddy' 'Rice']

Min price examples:
['6850.985817820893' '₹6,944.79' 'Rs. 1,857' '₹1,873.34' '5,878.62/-' ''
 '₹2,270.14' '₹1,961.63' '2277.53' '₹3,156.94' 'Rs. 1,866'
 '1989.0202736774606' 'INR 2,016' '₹3,391.89' '₹3,661.39' 'INR 6,054'
 '2172.99' '

In [134]:
# Price & MSP - Investigate Missing Mandi IDs

# Treat both NaN and blank/whitespace values as missing
missing_mandi = (
    price["mandi_id"].isna()
    | price["mandi_id"].astype("string").str.strip().eq("")
)

print("Total Price/MSP records:", len(price))
print("Missing mandi_id:", missing_mandi.sum())
print("Mandi IDs available:", (~missing_mandi).sum())

# ---------------------------------------------------------
# District availability among records with missing mandi_id
# ---------------------------------------------------------

missing_mandi_district = (
    price.loc[missing_mandi, "district"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

print("\nDistrict missing among records with missing mandi_id:")
print(missing_mandi_district.isna().value_counts())

print("\nNumber of missing-mandi records WITH district:",
      missing_mandi_district.notna().sum())

print("Number of missing-mandi records WITHOUT district:",
      missing_mandi_district.isna().sum())

# ---------------------------------------------------------
# District distribution
# ---------------------------------------------------------

print("\nDistricts for records with missing mandi_id:")
print(
    missing_mandi_district
    .value_counts(dropna=False)
)

# ---------------------------------------------------------
# Sample records
# ---------------------------------------------------------

print("\nSample records with missing mandi_id:")
display(
    price.loc[
        missing_mandi,
        ["record_id", "date", "mandi_id", "district", "crop_name",
         "min_price", "max_price", "modal_price", "msp"]
    ].head(20)
)

Total Price/MSP records: 12000
Missing mandi_id: 1235
Mandi IDs available: 10765

District missing among records with missing mandi_id:
district
False    1063
True      172
Name: count, dtype: int64

Number of missing-mandi records WITH district: 1063
Number of missing-mandi records WITHOUT district: 172

Districts for records with missing mandi_id:
district
<NA>           172
Fatehabad       92
Ferozepur       91
Ludhiana        91
Sirsa           85
Ambala          84
Bathinda        83
Jalandhar       82
Amritsar        81
Kurukshetra     79
Patiala         77
Hisar           75
Moga            74
Karnal          69
Name: count, dtype: Int64

Sample records with missing mandi_id:


,record_id,date,mandi_id,district,crop_name,min_price,max_price,modal_price,msp
19,PR000719,2026-04-29,None,Kurukshetra,Makka,"INR 1,878",2114.21,,"₹2,090.00"
26,PR011958,02/04/2026,None,Bathinda,Kanak,"₹2,215.53","2,450.04/-",2332.79,2275
29,PR008575,20/07/2026,None,Moga,Ganne,"₹3,509.61","Rs. 4,356",3932.8,
30,PR010458,2026/01/10,None,Bathinda,Kapas,"Rs. 5,627","6,650.37/-",6138.929109845614,6620
50,PR008186,01-25-2026,None,Moga,Rice,2005.68,"₹2,112.51","₹2,059.10",
52,PR004775,02-23-2026,None,Jalandhar,Gehun,"INR 2,097","Rs. 2,609","INR 2,353",2275
55,PR002947,13/05/2026,None,Patiala,Cotton,"INR 6,131",7320.96,"₹6,725.89","₹6,620.00"
62,PR000060,14/05/2026,None,Ferozepur,mustard,"5,811.32/-",6329.19,"₹6,070.26",5650
64,PR003239,03.01.2026,None,Karnal,चावल,"INR 1,872",2174.23,2023.3017368627486,"INR 2,183"
73,PR001077,07-22-2026,None,Kurukshetra,सरसों,5178.312104348875,,5757.03,5650


In [135]:
# ============================================================
# PRICE & MSP CLEANING
# ============================================================

price_clean = price.copy()

# ------------------------------------------------------------
# 1. Clean blank strings
# ------------------------------------------------------------

# Convert blank/whitespace-only strings to missing values
for col in ["record_id", "date", "mandi_id", "district", "crop_name",
            "min_price", "max_price", "modal_price", "msp"]:
    price_clean[col] = (
        price_clean[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


# ------------------------------------------------------------
# 2. Clean record_id
# ------------------------------------------------------------

price_clean["record_id"] = (
    price_clean["record_id"]
    .str.upper()
    .str.strip()
)


# ------------------------------------------------------------
# 3. Clean mandi_id
# ------------------------------------------------------------

def clean_mandi_id(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    if not value:
        return pd.NA

    # Remove separators
    value = (
        value
        .replace("-", "")
        .replace("_", "")
        .replace(" ", "")
    )

    # M012 → MANDI012
    if value.startswith("M") and not value.startswith("MANDI"):
        number = value[1:]
        if number.isdigit():
            return f"MANDI{int(number):03d}"

    # MANDI012 → MANDI012
    if value.startswith("MANDI"):
        number = value[5:]
        if number.isdigit():
            return f"MANDI{int(number):03d}"

    # 012 → MANDI012
    if value.isdigit():
        return f"MANDI{int(value):03d}"

    # Preserve unexpected values for investigation
    return value


price_clean["mandi_id"] = price_clean["mandi_id"].apply(clean_mandi_id)


# ------------------------------------------------------------
# 4. Clean district
# ------------------------------------------------------------

price_clean["district"] = (
    price_clean["district"]
    .astype("string")
    .str.strip()
    .str.title()
    .replace("", pd.NA)
)


# ------------------------------------------------------------
# 5. Preserve original crop names
# ------------------------------------------------------------

price_clean["crop_name_raw"] = price_clean["crop_name"]


# ------------------------------------------------------------
# 6. Standardize crop names
# ------------------------------------------------------------

crop_mapping = {
    # Wheat
    "Wheat": "Wheat",
    "wheat": "Wheat",
    "WHEAT": "Wheat",
    "GEHUN": "Wheat",
    "Gehun": "Wheat",
    "Kanak": "Wheat",
    "गेहूं": "Wheat",

    # Rice
    "Rice": "Rice",
    "rice": "Rice",
    "Paddy": "Rice",
    "paddy": "Rice",
    "Dhaan": "Rice",
    "धान": "Rice",
    "Chawal": "Rice",
    "चावल": "Rice",

    # Maize
    "Maize": "Maize",
    "maize": "Maize",
    "Corn": "Maize",
    "corn": "Maize",
    "Makka": "Maize",
    "Makki": "Maize",
    "मक्का": "Maize",

    # Mustard
    "Mustard": "Mustard",
    "mustard": "Mustard",
    "Sarso": "Mustard",
    "Sarson": "Mustard",
    "सरसों": "Mustard",

    # Cotton
    "Cotton": "Cotton",
    "cotton": "Cotton",
    "Kapas": "Cotton",
    "Kapas": "Cotton",
    "Narma": "Cotton",
    "कपास": "Cotton",

    # Sugarcane
    "Sugarcane": "Sugarcane",
    "sugarcane": "Sugarcane",
    "Ganna": "Sugarcane",
    "Ganne": "Sugarcane",
    "गन्ना": "Sugarcane",

    # Basmati
    "Basmati": "Basmati",
    "basmati": "Basmati",
}


price_clean["crop_name"] = (
    price_clean["crop_name"]
    .map(crop_mapping)
)


# ------------------------------------------------------------
# 7. Check for unmapped crop names
# ------------------------------------------------------------

unmapped_crops = price_clean.loc[
    price_clean["crop_name"].isna() &
    price_clean["crop_name_raw"].notna(),
    "crop_name_raw"
].unique()

print("Unmapped crop values:")
print(unmapped_crops)

print("\nNumber of unmapped crop values:", len(unmapped_crops))


# ------------------------------------------------------------
# 8. Clean price fields
# ------------------------------------------------------------

price_columns = [
    "min_price",
    "max_price",
    "modal_price",
    "msp"
]

for col in price_columns:

    price_clean[col] = (
        price_clean[col]
        .astype("string")
        .str.strip()

        # Remove currency labels/symbols
        .str.replace("₹", "", regex=False)
        .str.replace("Rs.", "", regex=False)
        .str.replace("INR", "", regex=False)

        # Remove commas
        .str.replace(",", "", regex=False)

        # Remove trailing /-
        .str.replace("/-", "", regex=False)

        # Remove remaining whitespace
        .str.strip()

        # Blank → missing
        .replace("", pd.NA)

        # Convert to numeric
        .pipe(pd.to_numeric, errors="coerce")
    )


# ------------------------------------------------------------
# 9. Preserve raw date
# ------------------------------------------------------------

price_clean["date_raw"] = price_clean["date"]


# ------------------------------------------------------------
# 10. Parse dates
# ------------------------------------------------------------

def parse_price_dates(series):
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")

    formats = [
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%d/%m/%Y",
        "%d.%m.%Y",
        "%m-%d-%Y",
        "%d-%b-%Y",
    ]

    for fmt in formats:
        mask = result.isna() & series.notna()

        parsed = pd.to_datetime(
            series.loc[mask],
            format=fmt,
            errors="coerce"
        )

        result.loc[mask] = parsed

    return result


price_clean["date"] = parse_price_dates(price_clean["date"])


# ------------------------------------------------------------
# 11. Final data types
# ------------------------------------------------------------

price_clean["record_id"] = price_clean["record_id"].astype("string")
price_clean["mandi_id"] = price_clean["mandi_id"].astype("string")
price_clean["district"] = price_clean["district"].astype("string")
price_clean["crop_name"] = price_clean["crop_name"].astype("string")
price_clean["crop_name_raw"] = price_clean["crop_name_raw"].astype("string")


# ------------------------------------------------------------
# 12. Basic validation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PRICE & MSP CLEANING VALIDATION")
print("=" * 60)

print("\nRaw rows:", len(price))
print("Cleaned rows:", len(price_clean))

print("\nMissing values:")
print(price_clean.isna().sum())

print("\nDuplicate rows:", price_clean.duplicated().sum())

print("\nUnique mandi IDs:", price_clean["mandi_id"].nunique())

print("\nDate range:")
print("Minimum:", price_clean["date"].min())
print("Maximum:", price_clean["date"].max())

print("\nPrice data types:")
print(price_clean[price_columns].dtypes)

print("\nCrop names after cleaning:")
print(price_clean["crop_name"].value_counts(dropna=False))

print("\nMissing mandi_id:", price_clean["mandi_id"].isna().sum())
print("Missing district:", price_clean["district"].isna().sum())

Unmapped crop values:
<StringArray>
[]
Length: 0, dtype: string

Number of unmapped crop values: 0

PRICE & MSP CLEANING VALIDATION

Raw rows: 12000
Cleaned rows: 12000

Missing values:
record_id           0
date                0
mandi_id         1235
district         1546
crop_name           0
min_price         598
max_price         598
modal_price       610
msp              2377
crop_name_raw       0
date_raw            0
dtype: int64

Duplicate rows: 0

Unique mandi IDs: 57

Date range:
Minimum: 2026-01-01 00:00:00
Maximum: 2026-09-09 00:00:00

Price data types:
min_price      Float64
max_price      Float64
modal_price    Float64
msp            Float64
dtype: object

Crop names after cleaning:
crop_name
Maize        2064
Sugarcane    2014
Mustard      2011
Cotton       1996
Wheat        1983
Rice         1676
Basmati       256
Name: count, dtype: Int64

Missing mandi_id: 1235
Missing district: 1546


In [136]:
# Check how district missingness changed

print("Raw district missing:", price["district"].isna().sum())

raw_blank_district = (
    price["district"]
    .astype("string")
    .str.strip()
    .eq("")
    .sum()
)

print("Raw blank district strings:", raw_blank_district)

print("Cleaned district missing:", price_clean["district"].isna().sum())

print("\nExpected cleaned missing:",
      price["district"].isna().sum() + raw_blank_district)

Raw district missing: 773
Raw blank district strings: 773
Cleaned district missing: 1546

Expected cleaned missing: 1546


In [137]:
# Check whether all available mandi IDs exist in the Mandi Master

master_mandi_ids = set(master_clean["mandi_id"].dropna())

price_mandi_ids = set(price_clean["mandi_id"].dropna())

invalid_mandi_ids = price_mandi_ids - master_mandi_ids

print("Unique mandi IDs in Price data:", len(price_mandi_ids))
print("Unique mandi IDs in Master:", len(master_mandi_ids))
print("Invalid mandi IDs:", len(invalid_mandi_ids))

if len(invalid_mandi_ids) > 0:
    print("\nInvalid mandi IDs:")
    print(sorted(invalid_mandi_ids))
else:
    print("\nAll available mandi IDs are valid.")

Unique mandi IDs in Price data: 57
Unique mandi IDs in Master: 57
Invalid mandi IDs: 0

All available mandi IDs are valid.


In [138]:
# Check district consistency between Price data and Mandi Master

district_check = price_clean.merge(
    master_clean[["mandi_id", "district"]],
    on="mandi_id",
    how="left",
    suffixes=("_price", "_master")
)

# Compare only rows where both districts are available
check = district_check[
    district_check["district_price"].notna() &
    district_check["district_master"].notna()
].copy()

check["district_match"] = (
    check["district_price"].str.strip().str.lower()
    == check["district_master"].str.strip().str.lower()
)

print("Rows with both districts available:", len(check))
print("District matches:", check["district_match"].sum())
print("District mismatches:", (~check["district_match"]).sum())

if (~check["district_match"]).sum() > 0:
    print("\nDistrict mismatches:")
    print(
        check.loc[
            ~check["district_match"],
            ["mandi_id", "district_price", "district_master"]
        ].drop_duplicates()
    )

Rows with both districts available: 8721
District matches: 495
District mismatches: 8226

District mismatches:
      mandi_id district_price district_master
1     MANDI028      Fatehabad           Hisar
2     MANDI011      Jalandhar         Patiala
3     MANDI012      Ferozepur         Patiala
4     MANDI013         Karnal         Patiala
5     MANDI001        Patiala        Ludhiana
...        ...            ...             ...
4386  MANDI034      Fatehabad          Ambala
4687  MANDI024      Fatehabad          Karnal
4786  MANDI051    Kurukshetra      Saharanpur
4865  MANDI021          Hisar       Ferozepur
5849  MANDI056        Patiala        Bareilly

[650 rows x 3 columns]


In [139]:
# Remove previous merge column if it already exists
if "district_master" in price_clean.columns:
    price_clean = price_clean.drop(columns=["district_master"])

# Add canonical district from Mandi Master
price_clean = price_clean.merge(
    master_clean[["mandi_id", "district"]].rename(
        columns={"district": "district_master"}
    ),
    on="mandi_id",
    how="left"
)

# Use Master district when mandi_id is available
price_clean["district_final"] = price_clean["district_master"].combine_first(
    price_clean["district"]
)

# Check results
print("Total rows:", len(price_clean))
print("District available:", price_clean["district_final"].notna().sum())
print("District still missing:", price_clean["district_final"].isna().sum())

print("\nDistrict source:")
print(
    price_clean.apply(
        lambda row: "Master"
        if pd.notna(row["district_master"])
        else ("Price" if pd.notna(row["district"]) else "Missing"),
        axis=1
    ).value_counts()
)

Total rows: 12000
District available: 11729
District still missing: 271

District source:
Master     9996
Price      1733
Missing     271
Name: count, dtype: int64


In [140]:
# Check for invalid price values

price_columns = ["min_price", "max_price", "modal_price", "msp"]

print("Negative values:")
for col in price_columns:
    print(col, (price_clean[col] < 0).sum())

print("\nZero values:")
for col in price_columns:
    print(col, (price_clean[col] == 0).sum())

print("\nPrice relationship checks:")

# Minimum should not be greater than maximum
invalid_min_max = (
    price_clean["min_price"].notna() &
    price_clean["max_price"].notna() &
    (price_clean["min_price"] > price_clean["max_price"])
)

# Modal should normally be between min and max
invalid_modal = (
    price_clean["modal_price"].notna() &
    price_clean["min_price"].notna() &
    price_clean["max_price"].notna() &
    (
        (price_clean["modal_price"] < price_clean["min_price"]) |
        (price_clean["modal_price"] > price_clean["max_price"])
    )
)

print("Min price > Max price:", invalid_min_max.sum())
print("Modal price outside Min-Max:", invalid_modal.sum())

Negative values:
min_price 0
max_price 0
modal_price 0
msp 0

Zero values:
min_price 0
max_price 0
modal_price 0
msp 0

Price relationship checks:
Min price > Max price: 0
Modal price outside Min-Max: 0


In [141]:
# Remove temporary column
price_clean = price_clean.drop(columns=["district_master"])

# Reorder columns
price_clean = price_clean[
    [
        "record_id",
        "date",
        "date_raw",
        "mandi_id",
        "district",
        "district_final",
        "crop_name",
        "crop_name_raw",
        "min_price",
        "max_price",
        "modal_price",
        "msp"
    ]
]

# Final validation
print("Final Price/MSP shape:", price_clean.shape)
print("\nColumns:")
print(price_clean.columns.tolist())

print("\nMissing values:")
print(price_clean.isna().sum())

print("\nDuplicate rows:", price_clean.duplicated().sum())

Final Price/MSP shape: (12000, 12)

Columns:
['record_id', 'date', 'date_raw', 'mandi_id', 'district', 'district_final', 'crop_name', 'crop_name_raw', 'min_price', 'max_price', 'modal_price', 'msp']

Missing values:
record_id            0
date                 0
date_raw             0
mandi_id          1235
district          1546
district_final     271
crop_name            0
crop_name_raw        0
min_price          598
max_price          598
modal_price        610
msp               2377
dtype: int64

Duplicate rows: 0


In [142]:
# Save cleaned Price/MSP data

PRICE_CLEAN_PATH = "/content/cleaned/price_and_msp_clean.csv"

price_clean.to_csv(PRICE_CLEAN_PATH, index=False)

print("Saved successfully:")
print(PRICE_CLEAN_PATH)

print("\nRows saved:", len(price_clean))
print("Columns saved:", len(price_clean.columns))

Saved successfully:
/content/cleaned/price_and_msp_clean.csv

Rows saved: 12000
Columns saved: 12


In [143]:
msp_missing_by_crop = price_clean.groupby("crop_name").agg(
    total_records=("msp", "size"),
    missing_msp=("msp", lambda x: x.isna().sum())
)

msp_missing_by_crop["missing_percentage"] = (
    msp_missing_by_crop["missing_msp"]
    / msp_missing_by_crop["total_records"]
    * 100
)

msp_missing_by_crop = msp_missing_by_crop.round(2)

print(msp_missing_by_crop)

           total_records  missing_msp  missing_percentage
crop_name                                                
Basmati              256         47.0               18.36
Cotton              1996        391.0               19.59
Maize               2064        401.0               19.43
Mustard             2011        402.0               19.99
Rice                1676        325.0               19.39
Sugarcane           2014        422.0               20.95
Wheat               1983        389.0               19.62


In [144]:
# Check whether missing MSP values can be matched
# using the same crop and date

msp_by_crop_date = price_clean.groupby(
    ["crop_name", "date"]
)["msp"].agg(
    total_records="size",
    available_msp=lambda x: x.notna().sum(),
    missing_msp=lambda x: x.isna().sum()
)

print(msp_by_crop_date.head(20))

print("\nDates where MSP is available:")
print((msp_by_crop_date["available_msp"] > 0).sum())

print("Dates where MSP is completely missing:")
print((msp_by_crop_date["available_msp"] == 0).sum())

                      total_records  available_msp  missing_msp
crop_name date                                                 
Basmati   2026-01-01              1            1.0          0.0
          2026-01-03              2            2.0          0.0
          2026-01-05              1            0.0          1.0
          2026-01-07              1            0.0          1.0
          2026-01-08              1            0.0          1.0
          2026-01-11              5            4.0          1.0
          2026-01-12              2            2.0          0.0
          2026-01-13              1            1.0          0.0
          2026-01-14              3            2.0          1.0
          2026-01-15              1            1.0          0.0
          2026-01-16              1            1.0          0.0
          2026-01-19              1            1.0          0.0
          2026-01-20              1            1.0          0.0
          2026-01-22              2     

In [145]:
# Check whether MSP is consistent within each crop + date

msp_consistency = price_clean[
    price_clean["msp"].notna()
].groupby(
    ["crop_name", "date"]
)["msp"].nunique()

print("Crop-date groups with exactly 1 MSP value:",
      (msp_consistency == 1).sum())

print("Crop-date groups with multiple MSP values:",
      (msp_consistency > 1).sum())

print("\nGroups with multiple MSP values:")

multiple_msp = msp_consistency[msp_consistency > 1]

print(multiple_msp)

Crop-date groups with exactly 1 MSP value: 1653
Crop-date groups with multiple MSP values: 0

Groups with multiple MSP values:
Series([], Name: msp, dtype: int64)


In [146]:
# Find the unique MSP for each crop + date

msp_lookup = (
    price_clean[price_clean["msp"].notna()]
    .groupby(["crop_name", "date"])["msp"]
    .first()
    .reset_index()
)

# Rename for merging
msp_lookup = msp_lookup.rename(columns={"msp": "msp_from_crop_date"})

# Add the lookup to our data
price_clean = price_clean.merge(
    msp_lookup,
    on=["crop_name", "date"],
    how="left"
)

# Count how many missing MSP values can be recovered
recoverable = (
    price_clean["msp"].isna() &
    price_clean["msp_from_crop_date"].notna()
)

not_recoverable = (
    price_clean["msp"].isna() &
    price_clean["msp_from_crop_date"].isna()
)

print("Missing MSP before:", price_clean["msp"].isna().sum())
print("Recoverable MSP:", recoverable.sum())
print("Not recoverable:", not_recoverable.sum())


Missing MSP before: 2377
Recoverable MSP: 2347
Not recoverable: 30


In [147]:
# Fill missing MSP using the unique MSP for the same crop and date

price_clean["msp_imputed"] = price_clean["msp"]

# Identify values that will be filled
msp_will_be_filled = (
    price_clean["msp"].isna() &
    price_clean["msp_from_crop_date"].notna()
)

# Fill them
price_clean.loc[msp_will_be_filled, "msp_imputed"] = (
    price_clean.loc[msp_will_be_filled, "msp_from_crop_date"]
)

# Create an audit flag
price_clean["msp_was_imputed"] = msp_will_be_filled

# Check result
print("MSP values before:", price_clean["msp"].notna().sum())
print("MSP values imputed:", price_clean["msp_was_imputed"].sum())
print("MSP values still missing:", price_clean["msp_imputed"].isna().sum())

MSP values before: 9623
MSP values imputed: 2347
MSP values still missing: 30


In [148]:
# Remove temporary lookup column
price_clean = price_clean.drop(columns=["msp_from_crop_date"])

# Replace original MSP with the cleaned/imputed MSP
price_clean["msp"] = price_clean["msp_imputed"]

# Remove temporary column
price_clean = price_clean.drop(columns=["msp_imputed"])

# Final MSP check
print("MSP missing:", price_clean["msp"].isna().sum())
print("MSP imputed:", price_clean["msp_was_imputed"].sum())
print("Total rows:", len(price_clean))

MSP missing: 30
MSP imputed: 2347
Total rows: 12000


In [149]:
# Final validation of Price/MSP dataset

print("========== FINAL PRICE/MSP VALIDATION ==========")

print("Rows:", len(price_clean))
print("Columns:", len(price_clean.columns))

print("\nDuplicate rows:", price_clean.duplicated().sum())

print("\nMissing values:")
print(price_clean.isna().sum())

print("\nMSP imputed:", price_clean["msp_was_imputed"].sum())

print("\nDate range:")
print(price_clean["date"].min(), "to", price_clean["date"].max())

print("\nUnique crops:", price_clean["crop_name"].nunique())
print("Unique mandis:", price_clean["mandi_id"].nunique())

print("\nPrice data types:")
print(price_clean[[
    "min_price",
    "max_price",
    "modal_price",
    "msp"
]].dtypes)

========== FINAL PRICE/MSP VALIDATION ==========
Rows: 12000
Columns: 13

Duplicate rows: 0

Missing values:
record_id             0
date                  0
date_raw              0
mandi_id           1235
district           1546
district_final      271
crop_name             0
crop_name_raw         0
min_price           598
max_price           598
modal_price         610
msp                  30
msp_was_imputed       0
dtype: int64

MSP imputed: 2347

Date range:
2026-01-01 00:00:00 to 2026-09-09 00:00:00

Unique crops: 7
Unique mandis: 57

Price data types:
min_price      Float64
max_price      Float64
modal_price    Float64
msp            Float64
dtype: object


In [150]:
# Save final cleaned Price/MSP dataset

PRICE_CLEAN_PATH = "/content/cleaned/price_and_msp_clean.csv"

price_clean.to_csv(PRICE_CLEAN_PATH, index=False)

print("Price/MSP dataset saved successfully.")
print("Path:", PRICE_CLEAN_PATH)
print("Rows saved:", len(price_clean))

Price/MSP dataset saved successfully.
Path: /content/cleaned/price_and_msp_clean.csv
Rows saved: 12000


In [151]:
import pandas as pd

TRANSPORT = "/content/track3_transport_logistics.csv"

transport = pd.read_csv(TRANSPORT)

print("Shape:", transport.shape)

print("\nColumns:")
print(transport.columns.tolist())

print("\nData types:")
print(transport.dtypes)

print("\nFirst 5 rows:")
display(transport.head())

print("\nMissing values:")
print(transport.isna().sum())

print("\nDuplicate rows:", transport.duplicated().sum())

Shape: (10400, 10)

Columns:
['trip_id', 'mandi_id', 'destination_warehouse', 'departure_time', 'arrival_time', 'transit_hours', 'distance', 'distance_unit', 'vehicle_no', 'driver_id']

Data types:
trip_id                  object
mandi_id                 object
destination_warehouse    object
departure_time           object
arrival_time             object
transit_hours            object
distance                 object
distance_unit            object
vehicle_no               object
driver_id                object
dtype: object

First 5 rows:


,trip_id,mandi_id,destination_warehouse,departure_time,arrival_time,transit_hours,distance,distance_unit,vehicle_no,driver_id
0,TRP006374,MANDI050,WH-Central,2026-04-30T20:08:22,01/05/2026 05:26,9.3,305.714532,miles,NaN,DRV264
1,TRP008869,MANDI029,WH-West,04-10-2026 04:15 AM,10/04/2026 09:51,NaN,263.5,km,NaN,DRV540
2,TRP000674,MANDI024,WH-South,18/07/2026 16:34,07-18-2026 10:46 PM,6.2,276.5,km,UP 50 BC 6882,DRV722
3,TRP000036,MANDI-042,WH-Central,08-13-2026 06:21 AM,14/08/2026,19.7,1070.7,km,RJ-52-CD-3274,DRV103
4,TRP008288,mandi_019,WH-South,07-30-2026 10:51 PM,07-31-2026 03:57 PM,17.1,728.9,km,DL-73-DF-1458,NaN



Missing values:
trip_id                     0
mandi_id                    0
destination_warehouse       0
departure_time              0
arrival_time             1053
transit_hours             518
distance                    0
distance_unit            1032
vehicle_no               1622
driver_id                1551
dtype: int64

Duplicate rows: 400


In [152]:
# Investigate duplicate Transport records

duplicates = transport[transport.duplicated(keep=False)].copy()

print("Duplicate rows:", transport.duplicated().sum())

print("\nRows involved in duplicate groups:", len(duplicates))

print("\nFirst duplicate groups:")
display(duplicates.head(20))

print("\nDuplicate trip IDs:")
print(duplicates["trip_id"].value_counts().head(20))

Duplicate rows: 400

Rows involved in duplicate groups: 800

First duplicate groups:


,trip_id,mandi_id,destination_warehouse,departure_time,arrival_time,transit_hours,distance,distance_unit,vehicle_no,driver_id
1,TRP008869,MANDI029,WH-West,04-10-2026 04:15 AM,10/04/2026 09:51,NaN,263.5,km,NaN,DRV540
3,TRP000036,MANDI-042,WH-Central,08-13-2026 06:21 AM,14/08/2026,19.7,1070.7,km,RJ-52-CD-3274,DRV103
21,TRP005795,MANDI020,WH-South,2026-06-21T11:39:44,NaN,23.6,1390.6,km,NaN,DRV947
27,TRP000040,mandi006,WH-South,24/01/2026 07:05,24-Jan-2026 22:23:42,NaN,877.8,km,UP24-BC-3811,DRV670
36,TRP008147,MANDI019,WH-North,28-May-2026 13:14:44,2026-05-29T05:32:44,16.3,551.0939399,miles,HR-30-BC-9997,DRV384
37,TRP006177,mandi036,WH-West,04/01/2026 18:01,2026-01-04 22:07:19,4.1,196.7 KM,NaN,pb 73-AB-4041,DRV497
39,TRP000034,MANDI057,WH-West,2026-08-13T05:21:48,13-Aug-2026 14:15:48,8.9,437.5,km,DL-65-AB-3380,DRV761
45,TRP001122,MANDI011,WH-South,05-01-2026 10:46 PM,NaN,16.9,819.7,km,RJ-78-BC-1965,DRV222
49,TRP000305,MANDI026,Export-Terminal,22/06/2026,NaN,7.6,422.6,km,hr 20-CD-6378,DRV161
50,TRP000624,MANDI048,Export-Terminal,2026-04-01 04:37:31,2026-04-01 12:55:31,8.3,417.5,km,NaN,DRV190



Duplicate trip IDs:
trip_id
TRP008482    2
TRP006429    2
TRP006164    2
TRP007458    2
TRP008179    2
TRP009897    2
TRP006682    2
TRP008538    2
TRP008077    2
TRP008206    2
TRP009784    2
TRP005999    2
TRP007059    2
TRP006502    2
TRP009239    2
TRP009150    2
TRP002361    2
TRP000312    2
TRP009418    2
TRP003101    2
Name: count, dtype: int64


In [153]:
# Remove exact duplicate transport records

print("Rows before removing duplicates:", len(transport))

transport_clean = transport.drop_duplicates().copy()

print("Rows after removing duplicates:", len(transport_clean))
print("Duplicates remaining:", transport_clean.duplicated().sum())

Rows before removing duplicates: 10400
Rows after removing duplicates: 10000
Duplicates remaining: 0


In [154]:
# Inspect transit time and distance values

print("========== TRANSIT HOURS ==========")
print(transport_clean["transit_hours"].dropna().astype(str).head(20).tolist())

print("\nUnique transit hour formats/examples:")
print(transport_clean["transit_hours"].dropna().astype(str).unique()[:30])

print("\n========== DISTANCE ==========")
print(transport_clean["distance"].dropna().astype(str).head(20).tolist())

print("\nUnique distance unit values:")
print(transport_clean["distance_unit"].value_counts(dropna=False))

print("\n========== NEGATIVE TRANSIT HOURS ==========")
transit_numeric = pd.to_numeric(
    transport_clean["transit_hours"]
    .astype("string")
    .str.extract(r"(-?\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

print("Negative transit values:", (transit_numeric < 0).sum())
print("Missing transit values:", transport_clean["transit_hours"].isna().sum())

========== TRANSIT HOURS ==========
['9.3', '6.2', '19.7', '17.1', '7.3', '14.9', '6.4', '12.0', '9.4', '9.5', '15.7', '5.3', '13.1', '13.5', '7.7', '4.7 hrs', '17.2', '23.2', '23.6', '5.3 hrs']

Unique transit hour formats/examples:
['9.3' '6.2' '19.7' '17.1' '7.3' '14.9' '6.4' '12.0' '9.4' '9.5' '15.7'
 '5.3' '13.1' '13.5' '7.7' '4.7 hrs' '17.2' '23.2' '23.6' '5.3 hrs' '6.0'
 '13.3' '4.8' '13.7 hrs' '21.1' '13.4' '20.1' '10.1' '17.8' '2.3']

========== DISTANCE ==========
['305.714532', '263.5', '276.5', '1070.7', '728.9', '417.0', '671.9', '362.6', '549.5', '424.8', '553.3', '798.0', '275.0', '716.3', '716.4', '1000.8', '316.5 KM', '486.0', '244.3', '608.7571687000001']

Unique distance unit values:
distance_unit
km       7511
miles    1497
NaN       992
Name: count, dtype: int64

========== NEGATIVE TRANSIT HOURS ==========
Negative transit values: 539
Missing transit values: 502


In [155]:
# Clean transit hours and distance

# -----------------------------
# 1. Clean transit hours
# -----------------------------

transport_clean["transit_hours_clean"] = pd.to_numeric(
    transport_clean["transit_hours"]
    .astype("string")
    .str.extract(r"(-?\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

# Negative transit times are invalid
transport_clean.loc[
    transport_clean["transit_hours_clean"] < 0,
    "transit_hours_clean"
] = pd.NA


# -----------------------------
# 2. Clean distance
# -----------------------------

# Extract numeric value
transport_clean["distance_numeric"] = pd.to_numeric(
    transport_clean["distance"]
    .astype("string")
    .str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

# Clean distance unit
transport_clean["distance_unit_clean"] = (
    transport_clean["distance_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Recover KM from distance text when unit is missing
distance_has_km = (
    transport_clean["distance"]
    .astype("string")
    .str.contains(r"km", case=False, na=False)
)

transport_clean.loc[
    transport_clean["distance_unit_clean"].isna() & distance_has_km,
    "distance_unit_clean"
] = "km"


# Convert all distances to KM
transport_clean["distance_km"] = transport_clean["distance_numeric"]

miles_mask = transport_clean["distance_unit_clean"].eq("miles")

transport_clean.loc[miles_mask, "distance_km"] = (
    transport_clean.loc[miles_mask, "distance_numeric"] * 1.60934
)


# -----------------------------
# 3. Check results
# -----------------------------

print("Transit hours missing:",
      transport_clean["transit_hours_clean"].isna().sum())

print("Negative transit hours:",
      (transport_clean["transit_hours_clean"] < 0).sum())

print("\nDistance missing:",
      transport_clean["distance_km"].isna().sum())

print("\nDistance units:")
print(transport_clean["distance_unit_clean"].value_counts(dropna=False))

print("\nSample cleaned values:")
display(
    transport_clean[
        [
            "transit_hours",
            "transit_hours_clean",
            "distance",
            "distance_unit",
            "distance_unit_clean",
            "distance_km"
        ]
    ].head(15)
)

Transit hours missing: 1041
Negative transit hours: 0

Distance missing: 0

Distance units:
distance_unit_clean
km       8503
miles    1497
Name: count, dtype: Int64

Sample cleaned values:


,transit_hours,transit_hours_clean,distance,distance_unit,distance_unit_clean,distance_km
0,9.3,9.3,305.714532,miles,miles,491.998625
1,NaN,<NA>,263.5,km,km,263.5
2,6.2,6.2,276.5,km,km,276.5
3,19.7,19.7,1070.7,km,km,1070.7
4,17.1,17.1,728.9,km,km,728.9
5,7.3,7.3,417.0,km,km,417.0
6,14.9,14.9,671.9,km,km,671.9
7,6.4,6.4,362.6,km,km,362.6
8,12.0,12.0,549.5,km,km,549.5
9,9.4,9.4,424.8,km,km,424.8


In [156]:
# Inspect departure and arrival timestamp formats

print("========== DEPARTURE TIME EXAMPLES ==========")
print(
    transport_clean["departure_time"]
    .astype("string")
    .unique()[:30]
)

print("\n========== ARRIVAL TIME EXAMPLES ==========")
print(
    transport_clean["arrival_time"]
    .dropna()
    .astype("string")
    .unique()[:30]
)

print("\nMissing departure times:",
      transport_clean["departure_time"].isna().sum())

print("Missing arrival times:",
      transport_clean["arrival_time"].isna().sum())

========== DEPARTURE TIME EXAMPLES ==========
<StringArray>
[ '2026-04-30T20:08:22',  '04-10-2026 04:15 AM',     '18/07/2026 16:34',
  '08-13-2026 06:21 AM',  '07-30-2026 10:51 PM',  '07-25-2026 08:53 PM',
  '2026-07-05 07:34:30',           '19/06/2026', '02-May-2026 10:12:36',
     '05/09/2026 10:36',     '02/05/2026 09:15', '02-Aug-2026 03:53:48',
 '13-Jul-2026 10:12:45',  '2026-06-20T11:25:40',     '09/09/2026 14:19',
  '2026-06-26 06:21:28',  '01-01-2026 11:16 AM',     '09/02/2026 05:25',
  '05-20-2026 08:31 PM',           '09/06/2026',  '01-12-2026 07:55 PM',
  '2026-06-21T11:39:44',     '08/07/2026 22:27', '20-Mar-2026 23:11:29',
           '14/05/2026',     '27/05/2026 20:11',  '2026-06-21T01:20:44',
     '24/01/2026 07:05',  '08-20-2026 08:06 PM',           '08/06/2026']
Length: 30, dtype: string

========== ARRIVAL TIME EXAMPLES ==========
<StringArray>
[    '01/05/2026 05:26',     '10/04/2026 09:51',  '07-18-2026 10:46 PM',
           '14/08/2026',  '07-31-2026 03:57 PM',    

In [157]:
# Convert departure and arrival times to datetime

transport_clean["departure_datetime"] = pd.to_datetime(
    transport_clean["departure_time"],
    format="mixed",
    errors="coerce",
    dayfirst=False
)

transport_clean["arrival_datetime"] = pd.to_datetime(
    transport_clean["arrival_time"],
    format="mixed",
    errors="coerce",
    dayfirst=False
)

# Check parsing results
print("Departure datetime missing:",
      transport_clean["departure_datetime"].isna().sum())

print("Arrival datetime missing:",
      transport_clean["arrival_datetime"].isna().sum())

print("\nSample parsed timestamps:")

display(
    transport_clean[
        [
            "departure_time",
            "departure_datetime",
            "arrival_time",
            "arrival_datetime"
        ]
    ].head(20)
)

Departure datetime missing: 0
Arrival datetime missing: 1006

Sample parsed timestamps:


,departure_time,departure_datetime,arrival_time,arrival_datetime
0,2026-04-30T20:08:22,2026-04-30 20:08:22,01/05/2026 05:26,2026-01-05 05:26:00
1,04-10-2026 04:15 AM,2026-04-10 04:15:00,10/04/2026 09:51,2026-10-04 09:51:00
2,18/07/2026 16:34,2026-07-18 16:34:00,07-18-2026 10:46 PM,2026-07-18 22:46:00
3,08-13-2026 06:21 AM,2026-08-13 06:21:00,14/08/2026,2026-08-14 00:00:00
4,07-30-2026 10:51 PM,2026-07-30 22:51:00,07-31-2026 03:57 PM,2026-07-31 15:57:00
5,07-25-2026 08:53 PM,2026-07-25 20:53:00,NaN,NaT
6,2026-07-05 07:34:30,2026-07-05 07:34:30,05/07/2026 22:28,2026-05-07 22:28:00
7,19/06/2026,2026-06-19 00:00:00,2026-06-19T22:01:06,2026-06-19 22:01:06
8,02-May-2026 10:12:36,2026-05-02 10:12:36,02/05/2026 22:12,2026-02-05 22:12:00
9,05/09/2026 10:36,2026-05-09 10:36:00,05/09/2026 20:00,2026-05-09 20:00:00


In [158]:
# Parse Transport timestamps using the formats found in the dataset

def parse_transport_datetime(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    formats = [
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",

        "%d/%m/%Y %H:%M:%S",
        "%d/%m/%Y %H:%M",

        "%m-%d-%Y %I:%M %p",
        "%m-%d-%Y %H:%M",

        "%d-%m-%Y %I:%M %p",
        "%d-%m-%Y %H:%M",

        "%d-%b-%Y %H:%M:%S",
        "%d-%b-%Y %H:%M",

        "%d/%m/%Y",
        "%m-%d-%Y",
        "%d-%b-%Y"
    ]

    for fmt in formats:
        try:
            return pd.to_datetime(value, format=fmt)
        except:
            continue

    return pd.NaT


# Parse departure and arrival times
transport_clean["departure_datetime"] = (
    transport_clean["departure_time"].apply(parse_transport_datetime)
)

transport_clean["arrival_datetime"] = (
    transport_clean["arrival_time"].apply(parse_transport_datetime)
)


# Check results
print("Departure datetime missing:",
      transport_clean["departure_datetime"].isna().sum())

print("Arrival datetime missing:",
      transport_clean["arrival_datetime"].isna().sum())

print("\nSample parsed timestamps:")

display(
    transport_clean[
        [
            "departure_time",
            "departure_datetime",
            "arrival_time",
            "arrival_datetime"
        ]
    ].head(20)
)

Departure datetime missing: 0
Arrival datetime missing: 1006

Sample parsed timestamps:


,departure_time,departure_datetime,arrival_time,arrival_datetime
0,2026-04-30T20:08:22,2026-04-30 20:08:22,01/05/2026 05:26,2026-05-01 05:26:00
1,04-10-2026 04:15 AM,2026-04-10 04:15:00,10/04/2026 09:51,2026-04-10 09:51:00
2,18/07/2026 16:34,2026-07-18 16:34:00,07-18-2026 10:46 PM,2026-07-18 22:46:00
3,08-13-2026 06:21 AM,2026-08-13 06:21:00,14/08/2026,2026-08-14 00:00:00
4,07-30-2026 10:51 PM,2026-07-30 22:51:00,07-31-2026 03:57 PM,2026-07-31 15:57:00
5,07-25-2026 08:53 PM,2026-07-25 20:53:00,NaN,NaT
6,2026-07-05 07:34:30,2026-07-05 07:34:30,05/07/2026 22:28,2026-07-05 22:28:00
7,19/06/2026,2026-06-19 00:00:00,2026-06-19T22:01:06,2026-06-19 22:01:06
8,02-May-2026 10:12:36,2026-05-02 10:12:36,02/05/2026 22:12,2026-05-02 22:12:00
9,05/09/2026 10:36,2026-09-05 10:36:00,05/09/2026 20:00,2026-09-05 20:00:00


In [159]:
# Calculate transit time from departure and arrival timestamps

transport_clean["calculated_transit_hours"] = (
    transport_clean["arrival_datetime"] -
    transport_clean["departure_datetime"]
).dt.total_seconds() / 3600


# Check negative calculated durations
negative_calculated = (
    transport_clean["calculated_transit_hours"] < 0
).sum()

# Check extremely long journeys
extreme_transit = (
    transport_clean["calculated_transit_hours"] > 72
).sum()

print("Calculated transit time missing:",
      transport_clean["calculated_transit_hours"].isna().sum())

print("Negative calculated transit times:",
      negative_calculated)

print("Transit times greater than 72 hours:",
      extreme_transit)

print("\nCalculated transit time statistics:")
print(
    transport_clean["calculated_transit_hours"].describe()
)

print("\nSample calculated transit times:")

display(
    transport_clean[
        [
            "departure_datetime",
            "arrival_datetime",
            "transit_hours_clean",
            "calculated_transit_hours"
        ]
    ].head(20)
)

Calculated transit time missing: 1006
Negative calculated transit times: 328
Transit times greater than 72 hours: 0

Calculated transit time statistics:
count    8994.000000
mean       12.947990
std         8.538882
min       -21.483333
25%         6.800000
50%        13.000000
75%        19.003264
max        47.625556
Name: calculated_transit_hours, dtype: float64

Sample calculated transit times:


,departure_datetime,arrival_datetime,transit_hours_clean,calculated_transit_hours
0,2026-04-30 20:08:22,2026-05-01 05:26:00,9.3,9.293889
1,2026-04-10 04:15:00,2026-04-10 09:51:00,<NA>,5.600000
2,2026-07-18 16:34:00,2026-07-18 22:46:00,6.2,6.200000
3,2026-08-13 06:21:00,2026-08-14 00:00:00,19.7,17.650000
4,2026-07-30 22:51:00,2026-07-31 15:57:00,17.1,17.100000
5,2026-07-25 20:53:00,NaT,7.3,NaN
6,2026-07-05 07:34:30,2026-07-05 22:28:00,14.9,14.891667
7,2026-06-19 00:00:00,2026-06-19 22:01:06,6.4,22.018333
8,2026-05-02 10:12:36,2026-05-02 22:12:00,12.0,11.990000
9,2026-09-05 10:36:00,2026-09-05 20:00:00,9.4,9.400000


In [160]:
# Create final transit hours
# Keep valid recorded values first.
# Use calculated duration only to recover missing/invalid values.

transport_clean["transit_hours_final"] = transport_clean["transit_hours_clean"]

# Valid calculated transit times
valid_calculated = (
    transport_clean["calculated_transit_hours"].notna()
    & (transport_clean["calculated_transit_hours"] >= 0)
)

# Fill only where the original transit time is missing/invalid
fill_mask = (
    transport_clean["transit_hours_final"].isna()
    & valid_calculated
)

transport_clean.loc[fill_mask, "transit_hours_final"] = (
    transport_clean.loc[fill_mask, "calculated_transit_hours"]
)

# Track which values were recovered
transport_clean["transit_hours_imputed"] = fill_mask


print("Original valid transit hours:",
      transport_clean["transit_hours_clean"].notna().sum())

print("Transit hours recovered from timestamps:",
      transport_clean["transit_hours_imputed"].sum())

print("Final missing transit hours:",
      transport_clean["transit_hours_final"].isna().sum())

print("Negative final transit hours:",
      (transport_clean["transit_hours_final"] < 0).sum())

print("\nFinal transit-hour statistics:")
print(transport_clean["transit_hours_final"].describe())

Original valid transit hours: 8959
Transit hours recovered from timestamps: 900
Final missing transit hours: 141
Negative final transit hours: 0

Final transit-hour statistics:
count       9859.0
mean     13.024829
std       6.434696
min            0.0
25%            7.6
50%           13.0
75%          18.45
max          44.31
Name: transit_hours_final, dtype: Float64


In [161]:
# Clean vehicle registration numbers

transport_clean["vehicle_no_raw"] = transport_clean["vehicle_no"]

transport_clean["vehicle_no_clean"] = (
    transport_clean["vehicle_no"]
    .astype("string")
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

# Convert empty strings to missing
transport_clean.loc[
    transport_clean["vehicle_no_clean"].eq(""),
    "vehicle_no_clean"
] = pd.NA

print("Original vehicle numbers missing:",
      transport_clean["vehicle_no"].isna().sum())

print("Clean vehicle numbers missing:",
      transport_clean["vehicle_no_clean"].isna().sum())

print("\nSample vehicle numbers:")

display(
    transport_clean[
        ["vehicle_no_raw", "vehicle_no_clean"]
    ].drop_duplicates().head(30)
)

Original vehicle numbers missing: 1560
Clean vehicle numbers missing: 1560

Sample vehicle numbers:


,vehicle_no_raw,vehicle_no_clean
0,NaN,<NA>
2,UP 50 BC 6882,UP50BC6882
3,RJ-52-CD-3274,RJ52CD3274
4,DL-73-DF-1458,DL73DF1458
5,UP97-BC-6135,UP97BC6135
6,PB-99-BC-2796,PB99BC2796
7,pb 48-DF-3590,PB48DF3590
8,DL55-BC-7556,DL55BC7556
9,PB 83 DF 2210,PB83DF2210
10,PB 53 AB 4593,PB53AB4593


In [162]:
# Final Transport dataset validation

print("========== TRANSPORT VALIDATION ==========")

print("\nRows:", len(transport_clean))
print("Columns:", len(transport_clean.columns))

print("\nDuplicate rows:",
      transport_clean.duplicated().sum())

print("\nMissing values:")
print(transport_clean.isna().sum())

print("\nTransit hours:")
print("Valid:", transport_clean["transit_hours_final"].notna().sum())
print("Missing:", transport_clean["transit_hours_final"].isna().sum())
print("Negative:",
      (transport_clean["transit_hours_final"] < 0).sum())

print("\nDistance:")
print("Missing KM:", transport_clean["distance_km"].isna().sum())
print("Negative KM:",
      (transport_clean["distance_km"] < 0).sum())

print("\nDistance units:")
print(transport_clean["distance_unit_clean"].value_counts(dropna=False))

print("\nVehicle numbers:")
print("Missing:", transport_clean["vehicle_no_clean"].isna().sum())

print("\nUnique mandis:",
      transport_clean["mandi_id"].nunique())

print("\nDestination warehouses:")
print(transport_clean["destination_warehouse"].value_counts())

print("\nFinal columns:")
print(transport_clean.columns.tolist())

========== TRANSPORT VALIDATION ==========

Rows: 10000
Columns: 21

Duplicate rows: 0

Missing values:
trip_id                        0
mandi_id                       0
destination_warehouse          0
departure_time                 0
arrival_time                1006
transit_hours                502
distance                       0
distance_unit                992
vehicle_no                  1560
driver_id                   1512
transit_hours_clean         1041
distance_numeric               0
distance_unit_clean            0
distance_km                    0
departure_datetime             0
arrival_datetime            1006
calculated_transit_hours    1006
transit_hours_final          141
transit_hours_imputed          0
vehicle_no_raw              1560
vehicle_no_clean            1560
dtype: int64

Transit hours:
Valid: 9859
Missing: 141
Negative: 0

Distance:
Missing KM: 0
Negative KM: 0

Distance units:
distance_unit_clean
km       8503
miles    1497
Name: count, dtype: Int64

Vehic

In [163]:
# Inspect Transport mandi IDs before cleaning

print("Unique raw mandi IDs:", transport_clean["mandi_id"].nunique())

print("\nSample mandi IDs:")
display(
    transport_clean["mandi_id"]
    .drop_duplicates()
    .sort_values()
    .head(100)
)

Unique raw mandi IDs: 342

Sample mandi IDs:


,mandi_id
152,001
150,002
874,003
1736,004
2926,005
1189,006
714,007
916,008
2292,009
906,010


In [164]:
# Normalize Transport mandi IDs

def clean_transport_mandi_id(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    # Remove common prefixes
    value = value.replace("MANDI", "")
    value = value.replace("M", "")

    # Keep only digits
    digits = "".join(ch for ch in value if ch.isdigit())

    if digits == "":
        return pd.NA

    # Convert to canonical format
    return f"MANDI{int(digits):03d}"


# Preserve original ID
transport_clean["mandi_id_raw"] = transport_clean["mandi_id"]

# Create cleaned ID
transport_clean["mandi_id"] = (
    transport_clean["mandi_id"]
    .apply(clean_transport_mandi_id)
)

print("Unique cleaned mandi IDs:",
      transport_clean["mandi_id"].nunique())

print("\nMissing cleaned mandi IDs:",
      transport_clean["mandi_id"].isna().sum())

print("\nSample normalized IDs:")
display(
    transport_clean[
        ["mandi_id_raw", "mandi_id"]
    ].drop_duplicates().head(100)
)

Unique cleaned mandi IDs: 57

Missing cleaned mandi IDs: 0

Sample normalized IDs:


,mandi_id_raw,mandi_id
0,MANDI050,MANDI050
1,MANDI029,MANDI029
2,MANDI024,MANDI024
3,MANDI-042,MANDI042
4,mandi_019,MANDI019
5,038,MANDI038
6,mandi041,MANDI041
7,023,MANDI023
8,mandi_033,MANDI033
9,MANDI054,MANDI054


In [165]:
# Check Transport mandi IDs against the cleaned Mandi Master

master_mandi_ids = set(master_clean["mandi_id"].dropna().unique())

transport_mandi_ids = set(
    transport_clean["mandi_id"].dropna().unique()
)

invalid_transport_ids = transport_mandi_ids - master_mandi_ids

print("Transport unique mandi IDs:",
      len(transport_mandi_ids))

print("Master unique mandi IDs:",
      len(master_mandi_ids))

print("Invalid Transport mandi IDs:",
      len(invalid_transport_ids))

if invalid_transport_ids:
    print("\nInvalid IDs:")
    print(sorted(invalid_transport_ids))
else:
    print("\nAll Transport mandi IDs match the cleaned Mandi Master. ✓")

Transport unique mandi IDs: 57
Master unique mandi IDs: 57
Invalid Transport mandi IDs: 0

All Transport mandi IDs match the cleaned Mandi Master. ✓


In [166]:
# Remove intermediate cleaning columns

transport_final = transport_clean.drop(
    columns=[
        "transit_hours_clean",
        "distance_numeric",
        "distance_unit_clean",
        "calculated_transit_hours"
    ]
).copy()

# Save cleaned Transport dataset
TRANSPORT_CLEAN_PATH = "/content/cleaned/transport_logistics_clean.csv"

transport_final.to_csv(
    TRANSPORT_CLEAN_PATH,
    index=False
)

print("Transport dataset saved successfully.")
print("Path:", TRANSPORT_CLEAN_PATH)
print("Shape:", transport_final.shape)

print("\nFinal columns:")
print(transport_final.columns.tolist())

Transport dataset saved successfully.
Path: /content/cleaned/transport_logistics_clean.csv
Shape: (10000, 18)

Final columns:
['trip_id', 'mandi_id', 'destination_warehouse', 'departure_time', 'arrival_time', 'transit_hours', 'distance', 'distance_unit', 'vehicle_no', 'driver_id', 'distance_km', 'departure_datetime', 'arrival_datetime', 'transit_hours_final', 'transit_hours_imputed', 'vehicle_no_raw', 'vehicle_no_clean', 'mandi_id_raw']


In [167]:
# Load Weather Sensors dataset

WEATHER = "/content/track3_weather_sensors.xlsx"

weather = pd.read_excel(WEATHER)

print("Shape:", weather.shape)

print("\nColumns:")
print(weather.columns.tolist())

print("\nData types:")
print(weather.dtypes)

print("\nMissing values:")
print(weather.isna().sum())

print("\nDuplicate rows:")
print(weather.duplicated().sum())

print("\nFirst 10 rows:")
display(weather.head(10))

Shape: (15000, 7)

Columns:
['sensor_id', 'timestamp', 'temperature', 'temp_unit', 'rainfall', 'rain_unit', 'humidity_percent']

Data types:
sensor_id            object
timestamp            object
temperature          object
temp_unit            object
rainfall            float64
rain_unit            object
humidity_percent    float64
dtype: object

Missing values:
sensor_id              0
timestamp           1555
temperature            0
temp_unit           2263
rainfall             791
rain_unit            791
humidity_percent    1500
dtype: int64

Duplicate rows:
0

First 10 rows:


,sensor_id,timestamp,temperature,temp_unit,rainfall,rain_unit,humidity_percent
0,SEN047,28/08/2026,68,f,43.80,mm,46.0
1,SEN034,2026-07-03 07:08:18 IST,85.3,°F,1.88,in,55.0
2,SEN020,NaN,18.9,Celsius,19.90,MM,NaN
3,SEN035,2026-07-09 11:08:41 UTC,71.8,Fahrenheit,0.91,inch,49.0
4,SEN048,2026-06-09 13:06:26 IST,39.8°C,NaN,43.90,mm,81.0
5,SEN035,17/03/2026,21.8°C,NaN,28.90,MM,47.0
6,SEN013,07-01-2026 05:48 PM,75,f,32.40,mm,43.0
7,SEN040,2026-06-12 09:52:51 IST,92.5,°F,NaN,NaN,92.0
8,SEN027,2026-06-02 16:00:04 UTC,22.1,°C,-48.10,mm,36.0
9,SEN034,NaN,30.3°C,NaN,NaN,NaN,51.0


In [168]:
# Inspect messy Weather values

print("========== TEMPERATURE ==========")
print(weather["temperature"].astype("string").drop_duplicates().head(30).to_list())

print("\nTemperature units:")
print(weather["temp_unit"].value_counts(dropna=False))

print("\n========== RAINFALL ==========")
print("Rainfall units:")
print(weather["rain_unit"].value_counts(dropna=False))

print("\nNegative rainfall values:",
      (weather["rainfall"] < 0).sum())

print("\nRainfall statistics:")
print(weather["rainfall"].describe())

print("\n========== TIMESTAMP ==========")
print("Sample timestamps:")
display(
    weather["timestamp"]
    .dropna()
    .drop_duplicates()
    .head(30)
)

print("\n========== HUMIDITY ==========")
print(weather["humidity_percent"].describe())

print("\nHumidity outside 0-100:",
      ((weather["humidity_percent"] < 0) |
       (weather["humidity_percent"] > 100)).sum())

========== TEMPERATURE ==========
['68', '85.3', '18.9', '71.8', '39.8°C', '21.8°C', '75', '92.5', '22.1', '30.3°C', '16.6', '102.2', '29.9', '66.9', '73.2', '102.4', '67.6', '29.8', '31.8°C', '29.1', '31.3', '30', '24.6°C', '97.9', '60.3', '37.8', '33.4', '28.3', '28.4°C', '33.6']

Temperature units:
temp_unit
NaN           2263
c             1733
°C            1683
C             1679
Celsius       1622
f             1551
°F            1539
F             1466
Fahrenheit    1464
Name: count, dtype: int64

========== RAINFALL ==========
Rainfall units:
rain_unit
mm             4459
MM             3038
millimeters    2916
inches         1285
in             1270
inch           1241
NaN             791
Name: count, dtype: int64

Negative rainfall values: 1518

Rainfall statistics:
count    14209.000000
mean        13.176203
std         20.879858
min        -50.000000
25%          0.980000
50%         10.000000
75%         29.900000
max         50.000000
Name: rainfall, dtype: float64

====

,timestamp
0,28/08/2026
1,2026-07-03 07:08:18 IST
3,2026-07-09 11:08:41 UTC
4,2026-06-09 13:06:26 IST
5,17/03/2026
6,07-01-2026 05:48 PM
7,2026-06-12 09:52:51 IST
8,2026-06-02 16:00:04 UTC
10,2026-08-23 07:42:11 IST
11,2026-09-08 14:45:54 UTC



========== HUMIDITY ==========
count    13500.000000
mean        62.298370
std         18.969047
min         30.000000
25%         46.000000
50%         62.000000
75%         79.000000
max         95.000000
Name: humidity_percent, dtype: float64

Humidity outside 0-100: 0


In [169]:
# Clean temperature and rainfall

# ---------- TEMPERATURE ----------

# Preserve original temperature
weather["temperature_raw"] = weather["temperature"]

# Extract numeric temperature
weather["temperature_numeric"] = pd.to_numeric(
    weather["temperature"]
    .astype("string")
    .str.extract(r"(-?\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

# Normalize temperature unit
weather["temp_unit_clean"] = (
    weather["temp_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Recover Celsius unit when °C appears inside the temperature value
temp_contains_c = (
    weather["temperature"]
    .astype("string")
    .str.contains("°c", case=False, na=False)
)

weather.loc[
    weather["temp_unit_clean"].isna() & temp_contains_c,
    "temp_unit_clean"
] = "c"

# Standardize unit names
weather["temp_unit_clean"] = weather["temp_unit_clean"].replace({
    "celsius": "c",
    "°c": "c",
    "fahrenheit": "f",
    "°f": "f"
})

# Convert Fahrenheit to Celsius
weather["temperature_c"] = weather["temperature_numeric"]

fahrenheit_mask = weather["temp_unit_clean"].eq("f")

weather.loc[fahrenheit_mask, "temperature_c"] = (
    (weather.loc[fahrenheit_mask, "temperature_numeric"] - 32) * 5 / 9
)


# ---------- RAINFALL ----------

weather["rainfall_raw"] = weather["rainfall"]

weather["rain_unit_clean"] = (
    weather["rain_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Standardize rainfall units
weather["rain_unit_clean"] = weather["rain_unit_clean"].replace({
    "millimeters": "mm",
    "inches": "in",
    "inch": "in"
})

# Negative rainfall is invalid
negative_rain = weather["rainfall"] < 0

weather.loc[negative_rain, "rainfall"] = pd.NA

# Convert rainfall to millimetres
weather["rainfall_mm"] = weather["rainfall"]

inches_mask = weather["rain_unit_clean"].eq("in")

weather.loc[inches_mask, "rainfall_mm"] = (
    weather.loc[inches_mask, "rainfall"] * 25.4
)


# ---------- VALIDATION ----------

print("Temperature unit distribution:")
print(weather["temp_unit_clean"].value_counts(dropna=False))

print("\nRainfall unit distribution:")
print(weather["rain_unit_clean"].value_counts(dropna=False))

print("\nTemperature missing:",
      weather["temperature_c"].isna().sum())

print("Rainfall missing:",
      weather["rainfall_mm"].isna().sum())

print("Negative rainfall after cleaning:",
      (weather["rainfall_mm"] < 0).sum())

print("\nTemperature °C statistics:")
print(weather["temperature_c"].describe())

print("\nRainfall mm statistics:")
print(weather["rainfall_mm"].describe())

Temperature unit distribution:
temp_unit_clean
c    8980
f    6020
Name: count, dtype: Int64

Rainfall unit distribution:
rain_unit_clean
mm      10413
in       3796
<NA>      791
Name: count, dtype: Int64

Temperature missing: 0
Rainfall missing: 2309
Negative rainfall after cleaning: 0

Temperature °C statistics:
count      15000.0
mean     27.430508
std       7.186954
min           15.0
25%      21.222222
50%           27.5
75%           33.6
max           40.0
Name: temperature_c, dtype: Float64

Rainfall mm statistics:
count    12691.000000
mean        25.029117
std         14.459714
min          0.000000
25%         12.600000
50%         24.900000
75%         37.592000
max         50.038000
Name: rainfall_mm, dtype: float64


In [170]:
# Clean Weather timestamps and convert everything to IST

def parse_weather_timestamp(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    # Detect timezone
    if value.endswith(" UTC"):
        value = value[:-4].strip()
        timezone = "UTC"
    elif value.endswith(" IST"):
        value = value[:-4].strip()
        timezone = "IST"
    else:
        timezone = "IST"

    # Try common formats
    formats = [
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
        "%Y-%m-%dT%H:%M:%S",

        "%d/%m/%Y %H:%M:%S",
        "%d/%m/%Y %H:%M",

        "%m-%d-%Y %I:%M %p",
        "%m-%d-%Y %H:%M",

        "%d-%m-%Y %I:%M %p",
        "%d-%m-%Y %H:%M",

        "%d/%m/%Y",
        "%m-%d-%Y"
    ]

    parsed = pd.NaT

    for fmt in formats:
        try:
            parsed = pd.to_datetime(value, format=fmt)
            break
        except:
            continue

    if pd.isna(parsed):
        return pd.NaT

    # Assign timezone
    if timezone == "UTC":
        parsed = parsed.tz_localize("UTC").tz_convert("Asia/Kolkata")
    else:
        parsed = parsed.tz_localize("Asia/Kolkata")

    return parsed


# Preserve original timestamp
weather["timestamp_raw"] = weather["timestamp"]

# Parse and convert to IST
weather["timestamp_ist"] = (
    weather["timestamp"].apply(parse_weather_timestamp)
)


print("Original timestamps missing:",
      weather["timestamp"].isna().sum())

print("Clean timestamps missing:",
      weather["timestamp_ist"].isna().sum())

print("\nSample converted timestamps:")

display(
    weather[
        ["timestamp_raw", "timestamp_ist"]
    ].dropna().head(25)
)

Original timestamps missing: 1555
Clean timestamps missing: 1917

Sample converted timestamps:


,timestamp_raw,timestamp_ist
0,28/08/2026,2026-08-28 00:00:00+05:30
1,2026-07-03 07:08:18 IST,2026-07-03 07:08:18+05:30
3,2026-07-09 11:08:41 UTC,2026-07-09 16:38:41+05:30
4,2026-06-09 13:06:26 IST,2026-06-09 13:06:26+05:30
5,17/03/2026,2026-03-17 00:00:00+05:30
6,07-01-2026 05:48 PM,2026-07-01 17:48:00+05:30
7,2026-06-12 09:52:51 IST,2026-06-12 09:52:51+05:30
8,2026-06-02 16:00:04 UTC,2026-06-02 21:30:04+05:30
10,2026-08-23 07:42:11 IST,2026-08-23 07:42:11+05:30
11,2026-09-08 14:45:54 UTC,2026-09-08 20:15:54+05:30


In [171]:
# Find non-missing timestamps that could not be parsed

bad_timestamps = weather[
    weather["timestamp"].notna() &
    weather["timestamp_ist"].isna()
]["timestamp"].astype("string")

print("Non-missing timestamps that failed to parse:",
      len(bad_timestamps))

print("\nExamples of failed timestamps:")

display(
    bad_timestamps
    .drop_duplicates()
    .head(50)
)

Non-missing timestamps that failed to parse: 362

Examples of failed timestamps:


,timestamp
55,12-Aug-2026 16:58:44
274,24-Jan-2026 02:18:48
304,06-Aug-2026 22:11:36
455,27-Feb-2026 01:30:40
549,24-Aug-2026 05:57:42
594,18-Jan-2026 17:29:13
628,06-Apr-2026 14:06:38
699,14-Feb-2026 01:44:16
715,03-Aug-2026 05:37:32
747,14-Aug-2026 08:20:36


In [172]:
# Fix remaining Weather timestamp format: DD-Mon-YYYY HH:MM:SS

def parse_weather_timestamp(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    # Detect timezone
    if value.endswith(" UTC"):
        value = value[:-4].strip()
        timezone = "UTC"
    elif value.endswith(" IST"):
        value = value[:-4].strip()
        timezone = "IST"
    else:
        timezone = "IST"

    formats = [
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
        "%Y-%m-%dT%H:%M:%S",

        "%d/%m/%Y %H:%M:%S",
        "%d/%m/%Y %H:%M",

        "%m-%d-%Y %I:%M %p",
        "%m-%d-%Y %H:%M",

        "%d-%m-%Y %I:%M %p",
        "%d-%m-%Y %H:%M",

        # Month-name formats
        "%d-%b-%Y %H:%M:%S",
        "%d-%b-%Y %H:%M",

        "%d/%m/%Y",
        "%m-%d-%Y",
        "%d-%b-%Y"
    ]

    parsed = pd.NaT

    for fmt in formats:
        try:
            parsed = pd.to_datetime(value, format=fmt)
            break
        except:
            continue

    if pd.isna(parsed):
        return pd.NaT

    # Convert to IST
    if timezone == "UTC":
        parsed = parsed.tz_localize("UTC").tz_convert("Asia/Kolkata")
    else:
        parsed = parsed.tz_localize("Asia/Kolkata")

    return parsed


# Recreate cleaned timestamp
weather["timestamp_ist"] = (
    weather["timestamp"].apply(parse_weather_timestamp)
)


# Validation
non_missing_original = weather["timestamp"].notna().sum()
failed_parse = (
    weather["timestamp"].notna()
    & weather["timestamp_ist"].isna()
).sum()

print("Original timestamps missing:",
      weather["timestamp"].isna().sum())

print("Non-missing timestamps:",
      non_missing_original)

print("Non-missing timestamps that failed to parse:",
      failed_parse)

print("Final timestamp missing:",
      weather["timestamp_ist"].isna().sum())

print("\nSample month-name timestamps:")

display(
    weather[
        weather["timestamp"].astype("string").str.contains(
            r"-[A-Za-z]{3}-", na=False
        )
    ][
        ["timestamp", "timestamp_ist"]
    ].head(15)
)

Original timestamps missing: 1555
Non-missing timestamps: 13445
Non-missing timestamps that failed to parse: 0
Final timestamp missing: 1555

Sample month-name timestamps:


,timestamp,timestamp_ist
55,12-Aug-2026 16:58:44,2026-08-12 16:58:44+05:30
274,24-Jan-2026 02:18:48,2026-01-24 02:18:48+05:30
304,06-Aug-2026 22:11:36,2026-08-06 22:11:36+05:30
455,27-Feb-2026 01:30:40,2026-02-27 01:30:40+05:30
549,24-Aug-2026 05:57:42,2026-08-24 05:57:42+05:30
594,18-Jan-2026 17:29:13,2026-01-18 17:29:13+05:30
628,06-Apr-2026 14:06:38,2026-04-06 14:06:38+05:30
699,14-Feb-2026 01:44:16,2026-02-14 01:44:16+05:30
715,03-Aug-2026 05:37:32,2026-08-03 05:37:32+05:30
747,14-Aug-2026 08:20:36,2026-08-14 08:20:36+05:30


In [173]:
# Validate cleaned Weather timestamps

print("========== TIMESTAMP VALIDATION ==========")

print("Missing timestamps:",
      weather["timestamp_ist"].isna().sum())

print("\nMinimum timestamp:")
print(weather["timestamp_ist"].min())

print("\nMaximum timestamp:")
print(weather["timestamp_ist"].max())

print("\nTimezone:")
print(weather["timestamp_ist"].dropna().iloc[0].tz)

print("\nUnique sensors:",
      weather["sensor_id"].nunique())

print("\nRecords per sensor:")
print(
    weather["sensor_id"]
    .value_counts()
    .describe()
)

print("\nSample final timestamps:")
display(
    weather[
        ["timestamp_raw", "timestamp_ist"]
    ].dropna().head(10)
)

========== TIMESTAMP VALIDATION ==========
Missing timestamps: 1555

Minimum timestamp:
2026-01-01 00:02:33+05:30

Maximum timestamp:
2026-09-09 23:38:15+05:30

Timezone:
Asia/Kolkata

Unique sensors: 51

Records per sensor:
count     51.000000
mean     294.117647
std       63.017029
min      246.000000
25%      276.000000
50%      286.000000
75%      299.000000
max      715.000000
Name: count, dtype: float64

Sample final timestamps:


,timestamp_raw,timestamp_ist
0,28/08/2026,2026-08-28 00:00:00+05:30
1,2026-07-03 07:08:18 IST,2026-07-03 07:08:18+05:30
3,2026-07-09 11:08:41 UTC,2026-07-09 16:38:41+05:30
4,2026-06-09 13:06:26 IST,2026-06-09 13:06:26+05:30
5,17/03/2026,2026-03-17 00:00:00+05:30
6,07-01-2026 05:48 PM,2026-07-01 17:48:00+05:30
7,2026-06-12 09:52:51 IST,2026-06-12 09:52:51+05:30
8,2026-06-02 16:00:04 UTC,2026-06-02 21:30:04+05:30
10,2026-08-23 07:42:11 IST,2026-08-23 07:42:11+05:30
11,2026-09-08 14:45:54 UTC,2026-09-08 20:15:54+05:30


In [174]:
# Validate humidity and sensor IDs

print("========== HUMIDITY VALIDATION ==========")

print("Missing humidity:",
      weather["humidity_percent"].isna().sum())

print("Humidity below 0:",
      (weather["humidity_percent"] < 0).sum())

print("Humidity above 100:",
      (weather["humidity_percent"] > 100).sum())

print("\nHumidity statistics:")
print(weather["humidity_percent"].describe())


print("\n========== SENSOR ID VALIDATION ==========")

print("Unique sensors:",
      weather["sensor_id"].nunique())

print("Missing sensor IDs:",
      weather["sensor_id"].isna().sum())

print("\nSample sensor IDs:")
print(weather["sensor_id"].drop_duplicates().head(20).to_list())

========== HUMIDITY VALIDATION ==========
Missing humidity: 1500
Humidity below 0: 0
Humidity above 100: 0

Humidity statistics:
count    13500.000000
mean        62.298370
std         18.969047
min         30.000000
25%         46.000000
50%         62.000000
75%         79.000000
max         95.000000
Name: humidity_percent, dtype: float64

========== SENSOR ID VALIDATION ==========
Unique sensors: 51
Missing sensor IDs: 0

Sample sensor IDs:
['SEN047', 'SEN034', 'SEN020', 'SEN035', 'SEN048', 'SEN013', 'SEN040', 'SEN027', 'SEN031', 'SEN045', 'SEN011', 'SEN004', 'SEN026', 'UNKNOWN', 'SEN032', 'SEN046', 'SEN021', 'SEN042', 'SEN043', 'SEN049']


In [175]:
# Final Weather validation and save

print("========== FINAL WEATHER VALIDATION ==========")

print("Rows:", len(weather))
print("Columns:", len(weather.columns))

print("\nDuplicate rows:", weather.duplicated().sum())

print("\nMissing values:")
print(weather.isna().sum())

print("\nTemperature:")
print("Missing:", weather["temperature_c"].isna().sum())
print("Min:", weather["temperature_c"].min())
print("Max:", weather["temperature_c"].max())

print("\nRainfall:")
print("Missing:", weather["rainfall_mm"].isna().sum())
print("Negative:", (weather["rainfall_mm"] < 0).sum())

print("\nHumidity:")
print("Missing:", weather["humidity_percent"].isna().sum())
print("Below 0:", (weather["humidity_percent"] < 0).sum())
print("Above 100:", (weather["humidity_percent"] > 100).sum())

print("\nTimestamp:")
print("Missing:", weather["timestamp_ist"].isna().sum())
print("Min:", weather["timestamp_ist"].min())
print("Max:", weather["timestamp_ist"].max())

print("\nSensors:",
      weather["sensor_id"].nunique())


# Save cleaned Weather dataset
WEATHER_CLEAN_PATH = "/content/cleaned/weather_sensors_clean.csv"

weather.to_csv(
    WEATHER_CLEAN_PATH,
    index=False
)

print("\nWeather dataset saved successfully.")
print("Path:", WEATHER_CLEAN_PATH)
print("Shape:", weather.shape)


========== FINAL WEATHER VALIDATION ==========
Rows: 15000
Columns: 16

Duplicate rows: 0

Missing values:
sensor_id                 0
timestamp              1555
temperature               0
temp_unit              2263
rainfall               2309
rain_unit               791
humidity_percent       1500
temperature_raw           0
temperature_numeric       0
temp_unit_clean           0
temperature_c             0
rainfall_raw            791
rain_unit_clean         791
rainfall_mm            2309
timestamp_raw          1555
timestamp_ist          1555
dtype: int64

Temperature:
Missing: 0
Min: 15.0
Max: 40.0

Rainfall:
Missing: 2309
Negative: 0

Humidity:
Missing: 1500
Below 0: 0
Above 100: 0

Timestamp:
Missing: 1555
Min: 2026-01-01 00:02:33+05:30
Max: 2026-09-09 23:38:15+05:30

Sensors: 51

Weather dataset saved successfully.
Path: /content/cleaned/weather_sensors_clean.csv
Shape: (15000, 16)
